## Import Statements and installations

In [226]:
from google.colab import files
import os # Define the files to be deleted and modified in google Colab
import shutil # Define the files to be deleted and modified in google Colab
import zipfile # Define the files to be deleted and modified in google Colab
from datetime import datetime #for date saving in file names.
from pytz import timezone
import pytz
import copy
from collections import defaultdict

## Global Variables

In [227]:
DEBUG = False #Used if need output from from methods for some methods.
PROCESS_SINGLE_AUTHOR = False # Used for testing purpose. When True then have to upload zip file of single author.
DISPLAY_CHARTS = False #when false then only will save the charts not show in colab.
tz_Berlin = pytz.timezone('Europe/Berlin') #Berlin time when saving date in the filename of xlsx
global CURRENT_DATE
CURRENT_DATE = datetime.now(tz_Berlin).strftime('%d%b%Y')  # for folder path wrt date


global network_list_Wertungen

global network_list_Relationen

global network_list_Wertungen_und_Relationen

global DOCUMENT_LIST_RELATIONEN
global DOCUMENT_LIST_WERTUNGEN
global DOCUMENT_LIST_WERTUNGEN_UND_RELATIONEN

DOCUMENT_LIST_RELATIONEN = []
DOCUMENT_LIST_WERTUNGEN = []
DOCUMENT_LIST_WERTUNGEN_UND_RELATIONEN = []


# Common Functions

### Fuction to delete all files

In [228]:
def action_delete_all_files(directory):
    '''
    Delete all files and folders in the specified directory,
    except the current notebook and mounted drives.
    '''
    try:
        current_file = os.path.basename(__file__) if '__file__' in globals() else None
        for filename in os.listdir(directory):
            file_path = os.path.join(directory, filename)

            # Skip the notebook file itself
            if current_file and filename == current_file:
                print(f"🛑 Skipped current notebook: {file_path}")
                continue

            # Skip Google Drive mount point if present
            if file_path.startswith("/content/drive"):
                print(f"🛑 Skipped Google Drive mount: {file_path}")
                continue

            # Delete files and folders
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)
                print(f"☠️ Deleted file: {file_path}")
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
                print(f"☠️ Deleted directory: {file_path}")
    except Exception as e:
        print(f"Error while deleting files: {e}")


### Function to upload just single author .zip

In [229]:
#Upload zip file intcontaining CAS XMI and XML
def upload_next_author_zip():

  #First delete files of previous author, if any.
  # Define the files to be deleted
  files_to_delete = ['TypeSystem.xml', 'CURATION_USER.xmi']

  # Path to the folder where the files are located
  content_folder_path = '/content/'

  # Delete the specified files if they exist
  for file_name in files_to_delete:
      file_path = os.path.join(content_folder_path, file_name)
      if os.path.isfile(file_path):
          os.remove(file_path)
          print(f"Deleted {file_path}")
      else:
          print(f"File {file_path} does not exist.")

  print(f"Deletion process completed for {files_to_delete} from path: {content_folder_path}.")
  print("\n")

  #Upload the ZIP file
  uploaded = files.upload()

  # Specify the extraction path
  content_folder_path = '/content/'

  # Extract the uploaded ZIP file to the specified folder
  for filename in uploaded.keys():
      # Ensure it's a zip file
      if filename.endswith('.zip'):
          with zipfile.ZipFile(filename, 'r') as zip_ref:
              zip_ref.extractall(content_folder_path)
              print(f'Extracted all files from {filename} to {content_folder_path}')
      else:
          print(f'{filename} is not a zip file')

  # Delete the ZIP file after extraction
  os.remove(filename)
  print(f'Deleted the ZIP file: {filename}')

### Functions of uploading zip file for all authors and extracting zip files and retun the list of folders.

In [230]:
#Extract files from from zip. for all authors "webanno*****export_curated_documents.zip"
def extract_nested_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as main_zip:
        # Extract the main zip
        main_zip.extractall(extract_to)

        # Iterate through each folder in the curation directory
        for folder_name in main_zip.namelist():
            if folder_name.endswith('.zip') and 'curation' in folder_name:
                nested_zip_path = os.path.join(extract_to, folder_name)

                # Extract the nested zip files
                with zipfile.ZipFile(nested_zip_path, 'r') as nested_zip:
                    nested_zip.extractall(os.path.dirname(nested_zip_path))

#returns the folder names as list in './curation'.
def get_folder_names(curation_path='./curation'):
    folder_names = [folder_name for folder_name in os.listdir(curation_path)
                    if os.path.isdir(os.path.join(curation_path, folder_name))]
    return folder_names



### Funtion to get metadata from the xmi file

In [231]:
#Gets Title from Metadata.
#Requires: CAS_XMI file
#Retuns: document_title from metadata
def document_metadata_info(typesystem_xml, doc_CAS_xmi, debug = False):
  # Get the DocumentMetaData type from the type system
  DocumentMetaData = typesystem_xml.get_type('de.tudarmstadt.ukp.dkpro.core.api.metadata.type.DocumentMetaData')
  # Retrieve the DocumentMetaData from the CAS Doc
  document_metadata_list = doc_CAS_xmi.select(DocumentMetaData)

  # Metadata list
  if document_metadata_list:
      document_metadata = document_metadata_list[0] # Access the first (and only) item in the list
      document_title = document_metadata.documentTitle  #Document Title

 #Calculated values from this cell.
  if debug:
    print("#################### DocumentMetaData #####################################")
    print(document_metadata_list)

  return document_title

# Daten Für Netzwerkanalysen generieren

In [232]:
!pip install dkpro-cassis
#!pip install spacy-sentiws
#!python -m spacy download de_core_news_sm
from cassis import *
from pathlib import *
import pprint

### x Relationen

In [233]:
def xRelationen(doc_CAS_xmi, exclude_self_evaluations ,debug = False):

  # Relationen ausgeben
  n= 0
  Relationen = {}
  for segment in doc_CAS_xmi.select('custom.Relation'):
      n = n+1

      if(debug):
        print("#################### Relationen #####################################")
        #print(segment)
        print(n, ". Relation: ", segment.label) # Natürlich-kulturell-Opposition oder Wertopposition
        print(segment.Governor.get_covered_text(), segment.Governor.label, segment.Governor.Name)
        print(segment.Dependent.get_covered_text(), segment.Dependent.label, segment.Dependent.Name)
        print("\n")
      if segment.Governor.Name is not None:
          Source = segment.Governor.Name.rsplit(": ")[1]
      else:
          Source = segment.Governor.get_covered_text()

      if segment.Dependent.Name is not None:
          Target = segment.Dependent.Name.rsplit(": ")[1]
      else:
          Target = segment.Dependent.get_covered_text()

      if (exclude_self_evaluations and Source==Target):  #17-11-24 remove self evaluations
        pass
      else:
        Relationen[n] = Source, Target, segment.label

  if(debug):
    print("Function Returns::::::: Relationen::::::")
    print(Relationen)
  return Relationen

### x Codierungen

In [234]:
def polarität_zu_int(polarität):
  """
  Gibt für verschiedene Codierungspolaritäten Integer aus, um Kompatibilität mit bisherigen Code (auf Wertungen bezogen) zu gewährleisten.

  Bsp: polarität == "sehr alt/traditionell" returns -1
  """
  match polarität:
    case "sehr althergebracht/traditionell" | "sehr alt/traditionell" | "sehr flach/oberflächlich" | "sehr kulturell/zivilisatorisch" | "sehr disharmonisch/fragmentiert" | "sehr krank/morbid":
      return -1
    case "althergebracht/traditionell" | "alt/traditionell" | "flach/oberflächlich" | "kulturell/zivilisatorisch" | "disharmonisch/fragmentiert" | "krank/morbid":
      return -1
    case "neuartig/modern" | "tief/tiefsinnig" | "natürlich/ursprünglich" | "harmonisch/ganzheitlich" | "gesund/heil":
      return 1
    case "sehr neuartig/modern" | "sehr tief/tiefsinnig" | "sehr natürlich/ursprünglich" | "sehr harmonisch/ganzheitlich" | "sehr gesund/heil":
      return 1
    case _:
      raise ValueError("Keine existierende Polarität: %s", polarität)


In [235]:
def xCodierungen(doc_CAS_xmi,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH,debug=False):
  # Codierungen auslesen
  Codierungen = {}
  j = 0
  # speichert Codierungen nach diesem Muster (analog dazu, wie Wertungen abgespeichert werden):
  # {1: (Codierende*r, codierte Entität, Art der Codierung, Polarität, Entitätentyp),
  #  2: (Codierende*r, codierte Entität, Art der Codierung, Polarität, Entitätentyp)}

  # ein segment == eine codierung
  for segment in doc_CAS_xmi.select('webanno.custom.AltNeuCodierung'):

      j = j+1
      #print(segment.get_covered_text())
      if segment.Polaritt_harmonisch:
        # polarität = segment.Polaritt_harmonisch # polarität als string abspeichern (zB "sehr alt/traditionell" statt -1)
        polarität = polarität_zu_int(segment.Polaritt_harmonisch)
      elif segment.Polaritt_gesund:
        # polarität = segment.Polaritt_gesund
        polarität = polarität_zu_int(segment.Polaritt_gesund)
      elif segment.Polaritt_natrlich:
        # polarität = segment.Polaritt_natrlich
        polarität = polarität_zu_int(segment.Polaritt_natrlich)
      elif segment.Polaritt_tief:
        # polarität = segment.Polaritt_tief
        polarität = polarität_zu_int(segment.Polaritt_tief)
      elif segment.Polaritt:
        # polarität = segment.Polaritt
        polarität = polarität_zu_int(segment.Polaritt)
      else:
        # polarität = None
        polarität = polarität_zu_int(None)

      codierte_entität = ""
      codierend = "Erzähler" # default Codierende*r ist Erzähler*in
      if(TRANSLATE_GERMAN_GRAPH_TO_ENGLISH):
        codierend = "Nameless Narrator1"
      if segment.Bezugsentitt: # d.h. wir ignorieren all jene Codierungen, bei denen keine codierte Entität o Codierende*r annotiert ist (weil schlecht darstellbar). Bei zB Roth werden dann zwei Codierungen nicht weiter berücksichtigt
        for e in segment.Bezugsentitt.elements:
            if e.role == "codierte Entität":
              ent_typ = e.target.label
              if e.target.Name:
                codierte_entität = e.target.Name.split(":")[1].strip()
              else:
                codierte_entität = e.target.get_covered_text()
            elif e.role == "Codierender": # wenn die Entität Codierender ist und nicht Codierter (gibts z.B. bei Roth)
              if e.target.Name:
                codierend = e.target.Name.split(":")[1].strip()
              else:
                codierend = e.target.get_covered_text()
        """
        # kompaktere Speicherungsart:
        # {codierte Entität A : [[Art der Codierung, Polarität, Codierende*r], [Art der Codierung2, Polarität2, Codierende*r2], ...]; codierte Entität B: []; ...}
        if codierte_entität in Codierungen: # wenn diese Entität bereits als key existiert, weil schon andere Codierung dazu abgespeichert wurde
          Codierungen[codierte_entität].append([segment.Art, polarität, codierend, ent_typ])
        else:
          Codierungen[codierte_entität] = [[segment.Art, polarität, codierend, ent_typ]]
        """
        if codierte_entität:
          Codierungen[j] = codierend, codierte_entität, segment.Art, polarität, ent_typ

  # testweise Ausgabe
  if(debug):
    print("======Codierungen:::::::::")
    for k,v in Codierungen.items():
      print(k, ": ", v)
  return Codierungen, j

### x Wertungen

In [236]:
#used in xWertungen
def xErzhlerwechsel(doc_CAS_xmi,debug=False):
  #All names, begin and end of Erzhlerwechsel
  Erzhlerwechsel_dict = {}
  index=0
  for segment in doc_CAS_xmi.select('webanno.custom.Erzhlerwechsel'):
    narrator_name = segment.NamedesneuenErzhlers.rsplit(": ")[1]
    Erzhlerwechsel_dict[index] = narrator_name,segment.begin, segment.end
    index+=1

  if (debug):
    print("xErzhlerwechsel:",Erzhlerwechsel_dict)

  return Erzhlerwechsel_dict

In [237]:
def xWertungen(doc_CAS_xmi,j, Erzhlerwechsel_dict, exclude_self_evaluations,debug=False):

  (default_name, default_begin, default_end) = Erzhlerwechsel_dict[0]  # First entry as default
  total_Erzhlerwechsel = len(Erzhlerwechsel_dict)
  next_Erzhlerwechsel = 1

  #j=0

  Wertungen = {}
  i=j # Nummerierung der Wertungen (= keys vom Wertungen-Dictionary) beginnt da, wo diejenige der Codierungen aufhört, sodass beide dictionaries
  # sich, wenn geüwnscht, mergen lassen und zusammen im Netzwerk darstellen lassen
  # s.u: Codierungen und/oder Wertungen anzeigen?

  # implement registration of "Erzählerwechsel" here to adopt default for "Wertender" below



  for segment in doc_CAS_xmi.select('webanno.custom.Wertung'):

      # Check if we are within bounds and update the default name based on segment's begin
      if next_Erzhlerwechsel < total_Erzhlerwechsel:
          (next_name, next_begin, next_end) = Erzhlerwechsel_dict[next_Erzhlerwechsel]
          if segment.begin >= next_begin:
              default_name = next_name
              next_Erzhlerwechsel += 1

      Wertender = default_name # default


      gew_Ent = None # default

      i = i+1

      #print(segment.TEST)
      #print(str(segment.TEST.elements))
      #print(segment.Label, ": ", segment.get_covered_text(), "; Hinsicht: ", segment.Wertungshinsicht, "; Polarität: ", segment.Polaritt)
      #if segment.Label == "implizite Wertung" or "explizite Wertung":
      Hinsicht = segment.Wertungshinsicht
      Polarität = segment.Polaritt
      Ironie = segment.Ironie

      if Polarität == "sehr positiv":
          Polarität_int = 1
      if Polarität == "positiv":
          Polarität_int = 1
      if Polarität == "sehr negativ":
          Polarität_int = -1
      if Polarität == "negativ":
          Polarität_int = -1

      # Wertung umkehren wenn Ironie vorhanden
      if Ironie == True:
          Polarität_int = -Polarität_int

      global abs_polaritPolarität
      if abs_polaritPolarität: #17-06-2025 positive and negative as +1 for evaluiations
        Polarität_int = abs(Polarität_int)

      try:
          for e in segment.TEST.elements:
              t= "test"
      except:
          if(debug):
            print("Segment ohne gew. Objekt")
          continue

      for e in segment.TEST.elements:

          if e.role == "Wertender":

              #print(e.role, ": ", e.target.get_covered_text(), "Typ: ", e.target.label, "Name: ", e.target.Name)
              if e.target.Name is not None:
                  Wertender = e.target.Name.rsplit(": ")[1]
              else:
                  Wertender = e.target.get_covered_text()

          if e.role == "gewertete Entität":
                  EntTyp = e.target.label
                  if e.target.Name is not None:
                      gew_Ent = e.target.Name.rsplit(": ")[1]
                  else:
                      gew_Ent = e.target.get_covered_text()

      # Wertung auf Kompositionsebene
      if segment.Wertung_auf_Kompositionsebene == 'Kompositionsebene':
        Wertender = "Autor"

      # Ausschluss von Fällen ohne Wertungsobjekt
      if(exclude_self_evaluations) and (Wertender == gew_Ent): #17-11-24 remove self evaluations
          pass
      elif gew_Ent is not None:
        if (Hinsicht=="eudämonistisch"):
          if not (EntTyp=="FIGUR" or EntTyp=="FIGURENGRUPPE"): #17-11-24: Exluded all eudämonistisch evaluation for FIGUR and FIGURENGRUPPE
            Wertungen[i] = Wertender, gew_Ent, Hinsicht, Polarität_int, EntTyp
        else:
          Wertungen[i] = Wertender, gew_Ent, Hinsicht, Polarität_int, EntTyp
  if(debug):
    print("====== Wertungen ::::::::::")
    for i, _wertungen in Wertungen.items():
      print(i,Wertungen)

  return Wertungen, i

### x Positive und negativ bewertete Entitäten

In [238]:
def xPositive_und_negativ_bewertete_Entitäten(Wertungen,debug=False):
  """
  Ent_Bewert speichert zu jeder bewerteten Entität ab, in welcher Hinsicht und mit welcher Polarität die bewertet wird,
  nach folgendem Muster (Beispiel):

  {'der Einsamkeit': {'ästhetisch': 0.5,
    'epistemisch': 0,
    'eudämonistisch': -0.5,
    'eudämonistisch Figur': 0,
    'moralisch': 0,
    'sozialer Status': 0,
    'sonstige': 0,
    'nicht spezifiziert': 0,
    'gesamt': 0.0,
    'gesamt_Erzähler': 0.0,
    'gesamt_Rest': 0,
    'Typ Entität': 'ABSTRAKTUM'}, ...}
  """

  Id = 0
  Ent_Bewert = {}

  keylist = ["ästhetisch", "epistemisch", "eudämonistisch", "eudämonistisch Figur", "moralisch",
                            "sozialer Status", "sonstige", "nicht spezifiziert", "gesamt", "gesamt_Erzähler", "gesamt_Rest", "Typ Entität"]

  for key, value in Wertungen.items():
      Wertender = value[0]
      Entität = value[1]
      Hinsicht = value[2]
      Polarität = value[3]
      EntTyp = value[4]
      #print(Wertungen.items())



      #if Wertender == "Erzähler":            #optional nur Erzählerwertungen
      if Entität in Ent_Bewert.keys():
          Ent_Bewert[Entität][Hinsicht] = Ent_Bewert[Entität][Hinsicht] + Polarität
          Ent_Bewert[Entität]["gesamt"] += Polarität
          if Wertender == "Erzähler" or Wertender.startswith("Namenloser Erzähler"): #28-06-2025 corrected
            try:
              Ent_Bewert[Entität]["gesamt_Erzähler"] = Ent_Bewert[Entität]["gesamt_Erzähler"] + Polarität
            except KeyError: # falls gesamt_Erzähler noch nicht initialisiert wurde
              Ent_Bewert[Entität]["gesamt_Erzähler"] = Polarität
          else: # Wertungen, die nicht vom Erzähler ausgehen
            try:
              Ent_Bewert[Entität]["gesamt_Rest"] = Ent_Bewert[Entität]["gesamt_Rest"] + Polarität
            except KeyError:
              Ent_Bewert[Entität]["gesamt_Rest"] = Polarität
      else: # wenn die entität noch nicht in Ent_Bewert ist
          empty_d = {}
          for i in keylist:
              try:
                empty_d[i] = 0
              except:
                continue
          Ent_Bewert[Entität] = empty_d
          Ent_Bewert[Entität][Hinsicht] = Polarität
          Ent_Bewert[Entität]["gesamt"] = Polarität
          if Wertender == "Erzähler" or Wertender.startswith("Namenloser Erzähler"): #28-06-2025 corrected
            Ent_Bewert[Entität]["gesamt_Erzähler"] = Polarität
          else:
            Ent_Bewert[Entität]["gesamt_Rest"] = Polarität
          Ent_Bewert[Entität]["Typ Entität"] = EntTyp

  if(debug):
    print('xPositive_und_negativ_bewertete_Entitäten::::::::::::::::::')
    print(Ent_Bewert)
  return Ent_Bewert

# Netzwerkanalyse

In [239]:
!pip install pyvis
!pip install cyjupyter
!pip install networkx

from pyvis.network import Network
from IPython.display import HTML
import pandas as pd
import networkx as nx
from collections import defaultdict

## Wertungen und Oppositionen als Kanten

In [240]:
#convert nicht spezifiziert to sonstige in evaluations.
def nichtspezifiziert_to_sonstige(Wertungen):
  for key, (source, target, eval_type, eval_value, ent_type) in Wertungen.items():
    if eval_type == "nicht spezifiziert":
      Wertungen[key] = source, target, "sonstige", eval_value, ent_type
  return Wertungen

In [241]:
#convertion of labels/types of opposition to encoding oppositions. for complete and split networks
def change_opposition_label(Relationen_german):
  new_dic_oppositions_ger = {
    "Alt-Neu-Opposition": "Codierung Opposition",
    "Wertopposition":"Value Opposition",
    "Traditionell-Modern-Opposition":"Codierung Opposition",
    "Harmonisch-disharmonisch-Opposition":	"Codierung Opposition",
    "Natürlich-kulturell-Opposition":"Codierung Opposition",
    "Gesund-krank-Opposition":"Codierung Opposition",
    "Tief-oberflächlich-Opposition":	"Codierung Opposition",
    "Sonstige Merkmalsopposition":"Feature Opposition",
    "Zugehörigkeit"	:"Belonging to",
    "Element von": "Element of",
    "Encoding Opposition": "Codierung Opposition"
  }

  # Create a new dictionary
  Relationen_new= {}


  for key, (source, target, opposition_type) in list(Relationen_german.items()):

    source = source
    target = target

    # Check if opposition_type exists in dic_German_English
    if opposition_type in new_dic_oppositions_ger:
        # Replace opposition_type with its English translation
        opposition_type_x = new_dic_oppositions_ger[opposition_type]
    else:
        # Keep the original if no translation is found
        opposition_type_x = opposition_type

    # Add the translated relation to the new dictionary
    Relationen_new[key] = (source, target, opposition_type_x)

  return Relationen_new


In [242]:
#Translation for complete and split networks
def translate_graph_to_english(Relationen_german,Wertungen_german):
  dic_oppositions_German_English = {
    "Alt-Neu-Opposition": "Traditional-Modern Opposition",
    "Wertopposition":"Value Opposition",
    "Traditionell-Modern-Opposition":"Traditional-Modern Opposition",
    "Harmonisch-disharmonisch-Opposition":	"Harmonious-Disharmonious Opposition",
    "Natürlich-kulturell-Opposition":"Natural-Cultural Opposition",
    "Gesund-krank-Opposition":"Healthy-ill Opposition",
    "Tief-oberflächlich-Opposition":	"Profound-Superficial Opposition",
    "Sonstige Merkmalsopposition":"Feature Opposition",
    "Zugehörigkeit"	:"Belonging to",
    "Element von": "Element of",
    "Codierung Opposition": "Encoding Opposition"
  }

  dic_evaluations_German_English = {
    "moralisch"	: "moral",
    "eudämonistisch"	: "eudaemonistic",
    "eudämonistisch Figur": "eudaemonistic character",
    "epistemisch" : "epistemic",
    "sozialer Status"	: "social status",
    "ästhetisch"	: "aesthetic",
    "sonstige"	: "other",
    "nicht spezifiziert": "not specified"
  }


  # Create a new dictionary for English translations
  Relationen_english = {}
  Wertungen_english = {}

  for key, (source, target, opposition_type) in list(Relationen_german.items()):
    if(source=="Namenloser Erzähler1"):
      source = "Nameless Narrator1"
    if(source=="Autor"):
      source = "Author"
    if(target=="Namenloser Erzähler1"):
      target = "Nameless Narrator1"
    if(target=="Autor"):
      target = "Author"

    # Check if opposition_type exists in dic_German_English
    if opposition_type in dic_oppositions_German_English:
        # Replace opposition_type with its English translation
        opposition_type_english = dic_oppositions_German_English[opposition_type]
    else:
        # Keep the original if no translation is found
        opposition_type_english = opposition_type

    # Add the translated relation to the new dictionary
    Relationen_english[key] = (source, target, opposition_type_english)


  for key, (source, target, eval_type, eval_value, ent_type) in Wertungen_german.items():
    if(source=="Namenloser Erzähler1"):
      source = "Nameless Narrator1"
    if(source=="Autor"):
      source = "Author"
    if(target=="Namenloser Erzähler1"):
      target = "Nameless Narrator1"
    if(target=="Autor"):
      target = "Author"

    if eval_type in dic_evaluations_German_English:
        # Replace eval_type with its English translation
        eval_type_english = dic_evaluations_German_English[eval_type]
    else:
        # Keep the original if no translation is found
        eval_type_english = eval_type
        print(eval_type_english)

    Wertungen_english[key] = (source, target, eval_type_english, eval_value, ent_type)

  return Relationen_english,Wertungen_english

In [244]:
def xCodierungen_und_oder_Wertungen_anzeigen(Codierungen, Wertungen, debug = False, anzeige_option = "y"):

  """
  anzeige_option == "c" --> nur Codierungen im Netzwerk
  anzeige_option == "w" --> nur Wertungen
  anzeige_option  == alles andere --> Wertungen und Codierungen
  Ausnahme: auf Ebene der Relationen werden immer auch nicht nur Wertoppositionen angezeigt, sondern
  auch zB nat-kult-Oppositionen
  """

  #anzeige_option = "y" # <------- set option. wenn geändert, danach noch mal alle mit x versehenen Abschnitte laufen lassen


  if anzeige_option == "w":
    df = pd.DataFrame.from_dict(Wertungen, orient = "index", columns=['Source', 'Target', 'Label', "Weight", "Type"])
  elif anzeige_option == "c":
    df = pd.DataFrame.from_dict(Codierungen, orient = "index", columns=['Source', 'Target', 'Label', "Weight", "Type"])
  else:
    combined_dict = {}
    combined_dict.update(Codierungen)
    combined_dict.update(Wertungen)
    df = pd.DataFrame.from_dict(combined_dict, orient = "index", columns=['Source', 'Target', 'Label', "Weight", "Type"])

  df["Weight"] = df["Weight"].astype(float)*5
  pd.set_option('display.max_rows', 5)
  if(debug):
    print(df)

  return df

In [ ]:
#New version. 18-12-2024 again updated on 17-06-2025

#network_list = [] : moved to main method as it does not have any effect here.

def Wertung_und_Oppositionen_als_Kanten(Wertungen, Relationen, Ent_Bewert,
                                        document_title, display_charts, debug=False,DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT=True,
                                        SIMPLIFY_GRAPHS = True,CREATE_SEPARATE_GRAPHS = True,
                                        CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION=True,RECOLOR_TO_GREY_EVALUATIONS=False, SIMPLIFY_OPPOSITIONS=True,ANTI_EDGE_OPPOSITION = True):




  keys_to_delete=[]
  Wertungen_temp = {}
  file_namex = ""
  end_path = ""
  f_name=""
  f_name = "complete_network"
  if CREATE_SEPARATE_GRAPHS:
    f_name = "split_network"


  global DOCUMENT_LIST_RELATIONEN
  global DOCUMENT_LIST_WERTUNGEN
  global DOCUMENT_LIST_WERTUNGEN_UND_RELATIONEN

  global network_list_Wertungen
  global network_list_Relationen
  global network_list_Wertungen_und_Relationen

  if(debug):
    print("====INPuT PARAMETERS FOR: Wertung_und_Oppositionen_als_Kanten=====")
    print("Wertungen",Wertungen)
    print("\n")
    print("Relationen", Relationen)
    print("\n")
    print("Ent_Bewert",Ent_Bewert)
    print("\n")

  #Added: 17-06-2025
  global set_opposition_weights
  set_opposition_weights = True
  if(set_opposition_weights):
    # Step 1: Group edges regardless of direction
    edge_groups = defaultdict(list)

    for _, (src, tgt, label) in Relationen.items():
        key = tuple(sorted([src, tgt]))  # Treat ('A', 'B') same as ('B', 'A')
        edge_groups[key].append((src, tgt, label))

    # Step 2: Create new dict with unique undirected edges, unified label, and weight
    new_Relationen = {}

    for key, edges in edge_groups.items():
        weight = len(edges)
        labels = {label for _, _, label in edges}
        final_label = labels.pop() if len(labels) == 1 else 'Opposition'

        # Pick a consistent direction: sort alphabetically or use the first one
        src, tgt, _ = edges[0]
        new_Relationen[(src, tgt)] = {
            'label': final_label,
            'weight': weight
        }

    # Optional: if you want to renumber keys like original Relationen
    final_Relationen = {
        i+1: (src, tgt, data['label'], data['weight'])
        for i, ((src, tgt), data) in enumerate(new_Relationen.items())
    }

    Relationen = copy.deepcopy(final_Relationen)
    del final_Relationen



  #Added: 24-11-2024
  if(DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT):
    # Define the set of opposition types to check
    opposition_types_to_remove = {"Element von", "Zugehörigkeit", "Element of", "Belonging to"}

    # Iterate and remove matching items
    for key, (source, target, opposition_type,weight) in list(Relationen.items()):
      if opposition_type in opposition_types_to_remove:
          del Relationen[key]



  if SIMPLIFY_OPPOSITIONS: #mod 18-12-2024 from SIMPLIFY_GRAPHS=TRUE
    if not set_opposition_weights: #17-06-2025 if req. Add default 0 weights to Relationen to avoid error

      # Temporary dictionary to store results with the same keys
      Relationen_temp = {}
      keys_to_delete=[]
      new_edge_connection = True
      flipped_once = False
      opposisition_appended_list = []

      #Two edges for oppositions = TRUE
      if(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION):
        for key, (source, target, opposition_type,weight) in Relationen.items():   #For each item in Relationen dictonary

          new_edge_connection = True
          flipped_once = False

          if not Relationen_temp: #temp dict is empty add first item
            Relationen_temp[key] = (source, target, opposition_type,weight)
            continue

          for key2, (source2, target2, opposition_type2,weight) in list(Relationen_temp.items()): #check for existing items in temp dict.
            if source == source2 and target == target2: #Check if new edge connection in the temp dict.
              new_edge_connection = False
              break
          if new_edge_connection: #Add new edge connection
            Relationen_temp[key] = (source, target, opposition_type,weight)
            continue
          else: #Found new_edge_connection = False: source == source2 and target == target2: Now check if the labels are same.
            if opposition_type not in opposition_type2: #if label is NOT part of label2. THEN: append label
                keys_to_delete += [key2]
                Relationen_temp[key] = (source, target, opposition_type+opposition_type2+"APPENDED1",weight)
            else: #if label IS part of or equal to label2. THEN: Flip
              flipped_once = True
              for key2, (source2, target2, opposition_type2,weight) in list(Relationen_temp.items()): #check for existing items in temp dict.
                if target == source2 and source == target2: # as flipped Check if new edge connection in the temp dict.
                  new_edge_connection = False
                  break
                else:
                  new_edge_connection = True
              if new_edge_connection and flipped_once: #Add new edge connection as flipped.
                Relationen_temp[key] = (target, source, opposition_type,weight)
              elif opposition_type == opposition_type2: #if label IS EQUAL to label2. THEN: delete old key and add new key.
                keys_to_delete += [key2]
                Relationen_temp[key] = (target, source, opposition_type,weight)
              else:
                keys_to_delete += [key2]
                Relationen_temp[key] = (target, source, opposition_type+opposition_type2+"APPENDED2",weight)


      else:#CREATE_TWO_OPPOSITE_EDGES_OPPOSITION=False: Single edge for oppositions #one_direction_simplify_opposition

        for key, (source, target, opposition_type,weight) in Relationen.items():
          for key2, (source2, target2, opposition_type2,weight) in list(Relationen_temp.items()):
            if(key2 in keys_to_delete):
              continue
            # Check if the value already exists in the temporary dictionary
            if source == source2 and target == target2:
                # Adjust eval_type and ent_type if they differ
                if opposition_type != opposition_type2:
                    opposition_type = "Opposition"
                Relationen_temp[key] = (source, target, opposition_type,weight)
                keys_to_delete += [key2]
            else:
                # Add a new entry
                Relationen_temp[key] = (source, target, opposition_type,weight)

          if not Relationen_temp:
            Relationen_temp[key] = (source, target, opposition_type,weight)






      #Clean up APPENDED labels for double edge oppositions and change it to "Opposition"
      for key2, (sourcex, targety, opposition_type2,weight) in list(Relationen_temp.items()):
        if "APPENDED1" in opposition_type2 or "APPENDED2" in opposition_type2:
          Relationen_temp[key2] = (sourcex, targety, "Opposition",weight)

      #Applies to both double edge oppositions and single edge opposition ie. CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION=TRUE and/or FALSE but always SIMPLIFY_GRAPHS=TRUE
      for key in keys_to_delete:
        try:
          del Relationen_temp[key]
        except:
          continue

      # Create a dictionary to store unique values
      unique_Relationen_temp_dict = {}
      for key in reversed(Relationen_temp):  # Iterate in reverse order
          value = Relationen_temp[key]
          if value not in unique_Relationen_temp_dict.values():
              unique_Relationen_temp_dict[key] = value

      # Reverse the dictionary back to original order
      unique_Relationen_temp_dict_sorted = dict(sorted(unique_Relationen_temp_dict.items()))

      # Reassign Relation
      Relationen = unique_Relationen_temp_dict_sorted.copy()



  #06-01-2025: Creates oppositions in opposite directions for all oppositions
  Relationen_temp = {}
  if ANTI_EDGE_OPPOSITION:
    for key, (source, target, opposition_type,weight) in Relationen.items():
      Relationen_temp[key] = (target, source, opposition_type,weight)
      Relationen_temp[-key] = (source, target,opposition_type,weight)
    Relationen = Relationen_temp.copy()


  #18-12-2024 simplify wetung
  if SIMPLIFY_GRAPHS:

    # Temporary dictionary to store results with the same keys
    Wertungen_temp = {}
    keys_to_delete=[]
    for key, (source, target, eval_type, eval_value, ent_type) in Wertungen.items():
        for key2, (source2, target2, eval_type2, eval_value2, ent_type2) in list(Wertungen_temp.items()):
          if(key2 in keys_to_delete):
            continue
          # Check if the value already exists in the temporary dictionary
          if source == source2 and target == target2:
              # Adjust eval_type and ent_type if they differ
              if eval_type != eval_type2:
                  eval_type = "Wertungen"
              if ent_type != ent_type2:
                  ent_type = "-"
              # Aggregate eval_value
              eval_value = eval_value + eval_value2
              #eval_value = eval_value * 2
              Wertungen_temp[key] = (source, target, eval_type, eval_value, ent_type)
              keys_to_delete += [key2]
          else:
              # Add a new entry
              Wertungen_temp[key] = (source, target, eval_type, eval_value, ent_type)

        if not Wertungen_temp:
          Wertungen_temp[key] = (source, target, eval_type, eval_value, ent_type)

    for key_list in keys_to_delete:
      del Wertungen_temp[key_list]


    # Reassign Wertungen
    Wertungen = Wertungen_temp.copy()


    if(debug):
      print("====INPuT PARAMETERS FOR GRAPHS: Wertung_und_Oppositionen_als_Kanten=====")
      print("Wertungen",Wertungen)
      print("\n")
      print("Relationen", Relationen)
      print("\n")
      print("Ent_Bewert",Ent_Bewert)
      print("\n")






  if(CREATE_SEPARATE_GRAPHS):
    graphs_list = [Wertungen,Relationen]
    flag_temp = False
    for item in graphs_list:
      if(flag_temp):
        Relationen = item
        Wertungen = {}
        save_dir_append_title = "Relationen"
        flag_temp = False
      else:
        Wertungen = item
        Relationen = {}
        save_dir_append_title = "Wertungen"
        flag_temp = True



      # Create a MultiDiGraph (directed graph with multiple edges)
      G = nx.MultiDiGraph()

      global split_pos_neg_evalustion_graphs #18-06-2025
      positive_eval_net = nx.MultiDiGraph()
      negative_eval_net = nx.MultiDiGraph()

      # Step 1: Add edges from Wertungen
      for key, (source, target, eval_type, eval_value, ent_type) in Wertungen.items():
          G.add_edge(source, target, label=eval_type, weight=eval_value, edge_type='evaluation')
          if split_pos_neg_evalustion_graphs:
            if eval_value > 0:
              positive_eval_net.add_edge(source, target, label=eval_type, weight=eval_value, edge_type='evaluation')
            else:
              abs_weight = abs(eval_value)
              negative_eval_net.add_edge(source, target, label=eval_type, weight=abs_weight, edge_type='evaluation')




      # Step 2: Add edges from Relationen (oppositions)
      opposition_count = defaultdict(int)

      for key, (source, target, opposition_type,weights) in Relationen.items():

          G.add_edge(source, target, label=opposition_type, weight=weights, edge_type='opposition')
          opposition_count[(source, target)] += 1

      #Above this MultiDiGraph construction is complete

      multigraphlist = [positive_eval_net, negative_eval_net, G] #18-06-2025
      for idx, graph in enumerate(multigraphlist):
          G = graph  # shadowing/reusing the name temporarily
          if idx==0:
            append_name_file_folder= "_positive"
            if save_dir_append_title != "Wertungen":
              continue
          elif idx ==1:
            append_name_file_folder= "_abs_negative"
            if save_dir_append_title != "Wertungen":
              continue
          else:
            append_name_file_folder= ""
          # legacy code that uses G





          #For HTML construction


          # Create a PyVis network optimized for Colab
          net = Network(directed=True, notebook=True, cdn_resources='in_line')

          # Step 3: Add nodes with sizes and colors based on Ent_Bewert and scale label size according to node degree
          for node in G.nodes():
              degree = G.degree(node)  # Number of edges connected to the node
              label_size = degree #* 2  # Scale the label size based on the degree
              if label_size < 10:
                label_size = 10  # Minimum label size
              #if node in Ent_Bewert: # and Ent_Bewert[node]['gesamt_Erzähler'] != 0:

              # node size to out_degree (number of evaluations)
              out_degrees = G.out_degree(node)
              node_size = abs(out_degrees) #* 15  # Scale node size for PyVis
              node_color = "grey"

              #else:
                  #node_size = 20  # Default size for other nodes
                  #node_color = 'blue'  # Default color for evaluating entities

              # Add node to PyVis network with label size based on node degree
              net.add_node(node, label=node, size=node_size, color=node_color, font={'size': label_size, 'bold': True})


          global abs_polaritPolarität
          # Step 4: Add edges for evaluations and oppositions
          for u, v, data in G.edges(data=True):
              label = data['label']
              edge_type = data['edge_type']

              # Check the weight of evaluation edges and color them based on positive or negative
              if edge_type == 'evaluation':
                  weight = data.get('weight', 0)  # Get weight of evaluation, default to 0 if not found
                  if(RECOLOR_TO_GREY_EVALUATIONS):
                    edge_color = 'grey'
                  else:
                    edge_color = 'green' if weight > 0 else 'red'  # Green for positive, red for negative
                    if append_name_file_folder== "_abs_negative":
                      edge_color = 'red'  # red for negative, have only red
                    if abs_polaritPolarität:
                      edge_color = 'blue' # blue for positive and negatives
                  line_width = abs(weight)
                  net.add_edge(u, v, label=label, color=edge_color, arrows='to', width=line_width, font={'size': 10})  # Edge labels size set to 10
              elif edge_type == 'opposition':
                weight = data.get('weight', 0)  # Get weight of oppostion, default to 0 if not found
                line_width = abs(weight)
                net.add_edge(u, v, label=label, color='orange',width=line_width, font={'size': 10})  # Edge labels size set to 10


          # Step 5: Set physics options to control node positioning based on opposition count
          # We'll use the repulsion force to spread nodes with more oppositions further apart
          net.set_options("""
          {
            "nodes": {
              "font": {
                "size": 12
              },
              "scaling": {
                "min": 10,
                "max": 30
              }
            },
            "edges": {
              "arrows": {
                "to": {
                  "enabled": true
                }
              },
              "width": 0.5,
              "scaling": {
                "min": 0.25,
                "max": 0.5,
                "label": {
                  "enabled": false
                }
              },
              "color": {
                "inherit": true,
                "type": "cubicBezier"
              },
              "smooth": {
                "enabled": true,
                "type": "dynamic"
              }
            },
            "physics": {
              "repulsion": {
                "nodeDistance": 200,
                "centralGravity": 0.1,
                "springLength": 200,
                "springConstant": 0.01,
                "damping": 0.09
              },
              "solver": "repulsion",
              "maxVelocity": 50,
              "minVelocity": 0.75,
              "barnesHut": {
                "gravitationalConstant": -8000,
                "springLength": 200,
                "springConstant": 0.005
              }
            },
            "interaction": {
              "hover": true
            }
          }
          """)







          file_namex = document_title + "_"+f_name + "_"+save_dir_append_title + append_name_file_folder+".html"

          # Define the subfolder path
          #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
          save_dir = f'{f_name}_{save_dir_append_title}{append_name_file_folder}_{CURRENT_DATE}'  # Create directory name based on time and category
          os.makedirs(save_dir, exist_ok=True)

          file_path = os.path.join(save_dir, file_namex)

          # Show the network inline in Google Colab
          net.show(file_path)
          if(display_charts):
            display(HTML(filename = file_path))

          # Save graph to GraphML format
          save_dir = f'{f_name}_graphml_{save_dir_append_title}{append_name_file_folder}_{CURRENT_DATE}'  # Create directory name based on time and category
          os.makedirs(save_dir, exist_ok=True)
          end_path=document_title + "_"+f_name+append_name_file_folder+".graphml"
          graphml_file_path = os.path.join(save_dir, end_path)
          #nx.write_graphml(G, graphml_file_path)
          nx.write_graphml(G, graphml_file_path)

          if idx == 2: #only for G.
            if(flag_temp):
              if (G.number_of_edges() > 0):
                DOCUMENT_LIST_WERTUNGEN.append(document_title)  #29-11-2024 used in the df of similarity score.
                network_list_Wertungen.append(G)
            else:
              if (G.number_of_edges() > 0):
                DOCUMENT_LIST_RELATIONEN.append(document_title)  #29-11-2024 used in the df of similarity score.
                network_list_Relationen.append(G)






  #replace else single graph
  else: #ELSE do NOT simplify

    # Create a MultiDiGraph (directed graph with multiple edges)
    G = nx.MultiDiGraph()

    # Step 1: Add edges from Wertungen
    for key, (source, target, eval_type, eval_value, ent_type) in Wertungen.items():
        G.add_edge(source, target, label=eval_type, weight=eval_value, edge_type='evaluation')
        #print(key, source, target, eval_type, eval_value, ent_type)

    # Step 2: Add edges from Relationen (oppositions)
    opposition_count = defaultdict(int)
    print("============Relationen============================",Relationen)


    for key, (source, target, opposition_type,wt) in Relationen.items():
        G.add_edge(source, target, label=opposition_type, edge_type='opposition')
        opposition_count[(source, target)] += 1

    # Create a PyVis network optimized for Colab
    net = Network(directed=True, notebook=True, cdn_resources='in_line')

    # Step 3: Add nodes with sizes and colors based on Ent_Bewert and scale label size according to node degree
    for node in G.nodes():

        # node size to out_degree (number of evaluations)
        out_degrees = G.out_degree(node)
        node_size = abs(out_degrees) #* 15  # Scale node size for PyVis
        node_color = "grey"

        # label size to degree
        degrees = G.degree(node)  # Number of edges connected to the node
        label_size = degrees #* 2  # Scale the label size based on the degree
        if label_size < 10:
          label_size = 10  # Minimum label size


        """
        if node in Ent_Bewert: # and Ent_Bewert[node]['gesamt_Erzähler'] != 0:

            total_eval = Ent_Bewert[node]['gesamt_Erzähler']
            #node_size = abs(total_eval) * 15  # Scale node size for PyVis
            #node_size = max(node_size, 10)  # Ensure node size is at least 10
            if total_eval > 0:
              node_color = 'grey'
            else:
              node_color = 'grey'
        else:
            node_size = 5  # Default size for other nodes
            node_color = 'grey'  # Default color for evaluating entities
        """

        # Add node to PyVis network with label size based on node degree
        net.add_node(node, label=node, size=node_size, color=node_color, font={'size': label_size, 'bold': True})

    # Step 4: Add edges for evaluations and oppositions
    for u, v, data in G.edges(data=True):
        label = data['label']
        edge_type = data['edge_type']

        # Check the weight of evaluation edges and color them based on positive or negative
        if edge_type == 'evaluation':
            weight = data.get('weight', 0)  # Get weight of evaluation, default to 0 if not found
            if(RECOLOR_TO_GREY_EVALUATIONS):
              edge_color = 'grey'
              #edge_color = 'orange'

            else:
              edge_color = "mediumseagreen" if weight > 0 else 'mediumblue'  # Green for positive, red for negative

            line_width = abs(weight)
            net.add_edge(u, v, label=label, color=edge_color, arrows='to', width=line_width, font={'size': 10})  # Edge labels size set to 10
        elif edge_type == 'opposition':
          net.add_edge(u, v, label=label, color='orange',width=2, font={'size': 10})  # Edge labels size set to 10


    # Step 5: Set physics options to control node positioning based on opposition count
    # We'll use the repulsion force to spread nodes with more oppositions further apart
    net.set_options("""
    {
      "nodes": {
        "font": {
          "size": 12
        },
        "scaling": {
          "min": 10,
          "max": 30
        }
      },
      "edges": {
        "arrows": {
          "to": {
            "enabled": true
          }
        },
        "width": 0.5,
        "scaling": {
          "min": 0.25,
          "max": 0.5,
          "label": {
            "enabled": false
          }
        },
        "color": {
          "inherit": true,
          "type": "cubicBezier"
        },
        "smooth": {
          "enabled": true,
          "type": "dynamic"
        }
      },
      "physics": {
        "repulsion": {
          "nodeDistance": 200,
          "centralGravity": 0.1,
          "springLength": 200,
          "springConstant": 0.01,
          "damping": 0.09
        },
        "solver": "repulsion",
        "maxVelocity": 50,
        "minVelocity": 0.75,
        "barnesHut": {
          "gravitationalConstant": -8000,
          "springLength": 200,
          "springConstant": 0.005
        }
      },
      "interaction": {
        "hover": true
      }
    }
    """)

    file_namex = document_title +"_"+ f_name+".html"

    # Define the subfolder path
    #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
    save_dir = f'{f_name}_{CURRENT_DATE}'  # Create directory name based on time and category
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(save_dir, file_namex)

    # Show the network inline in Google Colab
    net.show(file_path)
    if(display_charts):
      display(HTML(filename = file_path))

    # Save graph to GraphML format
    save_dir = f'{f_name}_graphml_{CURRENT_DATE}'  # Create directory name based on time and category
    os.makedirs(save_dir, exist_ok=True)
    end_path=document_title + "_"+f_name+".graphml"
    graphml_file_path = os.path.join(save_dir, end_path)
    #nx.write_graphml(G, graphml_file_path)
    nx.write_graphml(G, graphml_file_path)
    if (G.number_of_edges() > 0):
      DOCUMENT_LIST_WERTUNGEN_UND_RELATIONEN.append(document_title)  #29-11-2024 used in the df of similarity score.
      network_list_Wertungen_und_Relationen.append(G)

# Vorbereitung Multilayer

## x Codierungen und/oder Wertungen anzeigen?

## x Netzwerk mit Erzählerwertungen als Kanten (G)

In [247]:
# Networkx
!pip install pyvis
!pip install cyjupyter
!pip install networkx

from pyvis.network import Network
from IPython.display import HTML
import pandas as pd
import networkx as nx

In [248]:
def xCodierungen_und_Wertungen_als_Kanten(df, document_title,display_charts):

  # MultiDiGraph erstellen
  G = nx.MultiDiGraph()
  G_Fig = nx.MultiDiGraph()

  # benutzerdefinierte Farbpalette definieren
  color_palette = {"positive": "green", "negative": "red"}

  # Kanten aus DataFrame hinzufügen
  for i, row in df.iterrows():
      source = row["Source"]
      target = row["Target"]
      label = row["Label"]
      weight = row["Weight"]
      Typ = row["Type"]
      edge_color = color_palette["positive"] if weight >= 0 else color_palette["negative"]
      if (source == "Erzähler" or source.startswith("Namenloser Erzähler") or source.startswith("Nameless Narrator"))  and (Typ == "FIGUR" or Typ == "RAUM" or Typ == "BEREICH"):
        G.add_edge(source, target, key=i, label=label, weight=weight, width=weight, color=edge_color)
      if (source != "Erzähler" and not source.startswith("Namenloser Erzähler") and not source.startswith("Nameless Narrator"))  and (Typ == "FIGUR" or Typ == "RAUM" or Typ == "BEREICH"):
        G_Fig.add_edge(source, target, key=i, label=label, weight=weight, width=weight, color=edge_color)


  # Größe der Nodes
  scale=3 # Scaling the size of the nodes by 10*degree
  d = dict(G.degree)
  #print(d)

  #Updating dict
  d.update((x, scale*y) for x, y in d.items())
  #print(d)
  #Setting up size attribute
  nx.set_node_attributes(G,d,'size')

  # Netzwerk mit pyvis erstellen
  nt = Network(notebook=True, directed=True, cdn_resources='in_line')
  nt.from_nx(G)

  # Optionen für das Layout des Graphen festlegen
  nt.barnes_hut()
  nt.options.physics.enabled = True
  nt.options.edges.smooth = True

  # Node- und Kantenbeschriftungen aktivieren
  nt.show_buttons()
  nt.toggle_physics(True)


  # Kantenbreite und -farbe anpassen
  nt.options.edges.width = 1
  nt.options.edges.color = {"inherit":False}
  nt.options.edges.font = {"size": 25, "color": "#444444"}

  for node in nt.nodes:
      #node["color"] = "red"
      node['font'] = {'size': 60}


  file_namex = document_title + "_G_nx.html"

  # Define the subfolder path
  #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
  save_dir = f'Basic_narrator_evaluations_G_{CURRENT_DATE}'  # Create directory name based on time and category
  os.makedirs(save_dir, exist_ok=True)

  file_path = os.path.join(save_dir, file_namex)


  # Netzwerk anzeigen
  nt.show(file_path)
  # nur in colab
  if(display_charts):
    display(HTML(file_path))
  return (G, G_Fig)


## x Figurenwertungen (G_Fig)

In [249]:
#G.edges(data= True)
#G.nodes(data = True)
#nt.nodes

In [250]:
def xFigurenwertungen(G_Fig, document_title,display_charts):

  # Figurenwertungen
  # Größe der Nodes
  scale=3 # Scaling the size of the nodes by 10*degree
  d = dict(G_Fig.degree)
  #print(d)

  #Updating dict
  d.update((x, scale*y) for x, y in d.items())
  #print(d)
  #Setting up size attribute
  nx.set_node_attributes(G_Fig,d,'size')

  # Netzwerk mit pyvis erstellen
  nt = Network(notebook=True, directed=True, cdn_resources='in_line')
  nt.from_nx(G_Fig)

  # Optionen für das Layout des Graphen festlegen
  nt.barnes_hut()
  nt.options.physics.enabled = True
  nt.options.edges.smooth = True

  # Node- und Kantenbeschriftungen aktivieren
  nt.show_buttons()
  nt.toggle_physics(True)


  # Kantenbreite und -farbe anpassen
  nt.options.edges.width = 1
  nt.options.edges.color = {"inherit":False}
  nt.options.edges.font = {"size": 25, "color": "#444444"}

  for node in nt.nodes:
      #node["color"] = "red"
      node['font'] = {'size': 60}

  file_namex = document_title + "_G_Fig_nx.html"

  # Define the subfolder path
  #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
  save_dir = f'Basic_character_evaluations_G_Fig_nx_{CURRENT_DATE}'  # Create directory name based on time and category
  os.makedirs(save_dir, exist_ok=True)

  file_path = os.path.join(save_dir, file_namex)


  # Netzwerk anzeigen
  nt.show(file_path)
  # nur in colab
  if(display_charts):
    display(HTML(file_path))
  #return (G_Fig)

## x Netzwerk mit annotierten Oppositionen als Kanten (G2)

In [251]:


def xNetzwerk_mit_annotierten_oppositionen_als_kanten(Relationen, i, document_title,display_charts):
  # Achtung: Edges, die übereinanderliegen werden nicht dargestellt

  # MultiDiGraph erstellen
  G2 = nx.MultiDiGraph()

  # benutzerdefinierte Farbpalette definieren
  color_palette = {"Wertopposition": "green", "else": "red"}


  # Kanten aus DataFrame hinzufügen
  for key, value in Relationen.items():
      source = value[0]
      target = value[1]
      label = value[2]
      edge_color = color_palette["Wertopposition"] if label == "Wertopposition" else color_palette["else"]
      G2.add_edge(source, target, key=i, label=label, color=edge_color)

  # Größe der Nodes
  scale=10 # Scaling the size of the nodes by 10*degree
  d = dict(G2.degree)
  #print(d)

  #Updating dict
  d.update((x, scale*y) for x, y in d.items())
  #print(d)
  #Setting up size attribute
  nx.set_node_attributes(G2,d,'size')

  # Netzwerk mit pyvis erstellen
  nt = Network(notebook=True, directed=True, layout="hierarchical", cdn_resources='in_line')
  #nt.show_buttons(filter_=["physics"])

  nt.from_nx(G2)

  # Optionen für das Layout des Graphen festlegen
  nt.barnes_hut()
  nt.options.physics.enabled = True
  nt.options.edges.smooth = True

  # Node- und Kantenbeschriftungen aktivieren
  nt.show_buttons()
  nt.toggle_physics(False)


  # Kantenbreite und -farbe anpassen
  nt.options.edges.width = 1
  nt.options.edges.color = {"inherit":False}
  nt.options.edges.font = {"size": 10, "color": "#444444"}




  file_namex = document_title + "_G2_nx.html"

  # Define the subfolder path
  #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
  save_dir = f'Basic_opposition_networks_G2_nx_{CURRENT_DATE}'  # Create directory name based on time and category
  os.makedirs(save_dir, exist_ok=True)

  file_path = os.path.join(save_dir, file_namex)


  # Netzwerk anzeigen
  nt.show(file_path)
  # nur in colab
  if(display_charts):
    display(HTML(file_path))
  return (G2)


# Multilayer Network Analysis

### x Matplotlib

Voraussetzung: mit x markierte Abschnitte sind in aktueller Sitzung durchgelaufen

In [252]:
def update_Relationen_oppositions(Relationen):

  # all diejenigen Entitäten, die zu anderen Relationen gehören als Wertoppositionen, in entsprechende Listen speichern

  nat_kult_Oppositionen = [] # alle natürlich-kulturell Oppositionen
  alt_neu_Oppositionen = []
  flach_tief_Oppositionen = []
  disharm_harm_Oppositionen = []
  krank_gesund_Oppositionen = []

  for key, value in Relationen.items():
    if value[2] != "Wertopposition":
      match value[2]: # Relationsart
        case 'Alt-Neu-Opposition':
          alt_neu_Oppositionen.append([value[0], value[1]])
        case 'Natürlich-kulturell-Opposition':
          nat_kult_Oppositionen.append([value[0], value[1]])
        case 'Harmonisch-disharmonisch-Opposition':
          disharm_harm_Oppositionen.append([value[0], value[1]])
        case 'Tief-oberflächlich-Opposition':
          flach_tief_Oppositionen.append([value[0], value[1]])
        case 'Gesund-krank-Opposition':
          krank_gesund_Oppositionen.append([value[0], value[1]])
        case _:
          #raise ValueError("Keine existierende Opposition: %s", value[2])
          print ("def update_Relationen_oppositions(Relationen): Keine existierende Opposition: %s", value[2])
  return (  nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen)

In [253]:
def update_Codierungen_categories(Codierungen):
  # all diejenigen Entitäten, die codiert werden, zusammen mit der codierenden Entität speichern
  alt_neu = []
  flach_tief = []
  kult_nat = []
  disharm_harm = []
  krank_gesund = []

  for key, value in Codierungen.items():
    match value[2]: # die Codierungsart
      case 'Gesund-Krank':
        krank_gesund.append([value[0], value[1]])
      case 'Natürlich-Kulturell':
        kult_nat.append([value[0], value[1]])
      case 'Tief-Oberflächlich':
        flach_tief.append([value[0], value[1]])
      case 'Harmonisch-Disharmonisch':
        disharm_harm.append([value[0], value[1]])
      case 'Alt-Neu':
        alt_neu.append([value[0], value[1]])
      case 'Traditionell-Modern':
        alt_neu.append([value[0], value[1]])
      case _:
        raise ValueError("Keine existierende Codierungsart: %s", value[2])

  return(alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund)

In [254]:
def update_bewertet_categories(Ent_Bewert):
  # Listen mit all jenen Entitäten, die entweder vom Erzähler o einzelnen Figuren pos/neg/beides bewertet werden
  # Ent_Bewert[Entität]["gesamt_Erzähler"] speichert die addierten Wertungen des Erzählers,
  # Ent_Bewert[Entität]["gesamt_Rest"] die addierten Wertungen der Figuren.
  # --> für die Knotensymbolauswahl
  Figuren_pos_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Rest'] > 0]
  Erzähler_pos_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Erzähler'] > 0]
  Figuren_neutr_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Rest'] == 0]
  Erzähler_neutr_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Erzähler'] == 0]
  Figuren_neg_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Rest'] < 0]
  Erzähler_neg_bewertet = [k for k,v in Ent_Bewert.items() if v['gesamt_Erzähler'] < 0]
  return(Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet )

In [255]:
def update_nodelabels(G, G_Fig, G2, debug):
  ## Create Node Labels
  g_dict = nx.to_dict_of_dicts(G)
  g_dict_Fig = nx.to_dict_of_dicts(G_Fig)
  g_dict_Rel = nx.to_dict_of_dicts(G2)

  g_dict_gesamt = g_dict|g_dict_Fig|g_dict_Rel

  #print(g_dict)
  node_labels = {}
  for key in g_dict_gesamt:
    #print(key)
    node_labels[key] = str(key)
  if(debug):
    print(node_labels)



  """size=5
  for node in nt.nodes:
      #node["color"] = "red"
      node['font'] = {'size': 10}"""
  return(node_labels)

In [256]:
!apt-get install -y fonts-liberation
import matplotlib.font_manager as fm

!apt-get install python3-dev
#!pip install --upgrade setuptools #Salmoon: Why needed and where used ?? commented.
#!pip install matplotlib numpy
!pip install mpl_toolkits
from mpl_toolkits.mplot3d.axes3d import Axes3D

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-liberation is already the newest version (1:1.07.4-11).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3-dev is already the newest version (3.10.6-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
ERROR: Could not find a version that satisfies the requirement mpl_toolkits (from versions: none)
ERROR: No matching distribution found for mpl_toolkits


In [257]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from mpl_toolkits.mplot3d.art3d import Patch3DCollection
from matplotlib.patches import Arrow
from matplotlib.collections import PatchCollection
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

import matplotlib.patches as mpatches


def codierung_farbe(coords, alt_neu_coords, flach_tief_coords, kult_nat_coords, disharm_harm_coords, krank_gesund_coords):
  """
  Bestimmt Farbe basierend auf Codierungsart (oder Relationsart).

  coords: Koordinaten von codierender und codierter Entität (oder Koordinaten der zur Relation gehörenden Entitäten)
  alt_neu_coords etc: Listen an Koordinaten von alt/neu codierende u als alt/neu codierte Entität (oder alt_neu_opps_coords)
  Diese Listen werden in der Klasse LayeredNetworkGraph erstellt.

  Achtung: Nach Polarität wird nicht differenziert (würde unübersichtlich).
  D.h. sowohl eine als gesund codierte als auch eine als krank codierte Entität
  hätten einen magenta Pfeil, der auf sie zeigt, weil sie zur selben Codierungsart gehören ("gesund-krank").
  """
  if coords in alt_neu_coords:
    return "darkblue"
  elif coords in flach_tief_coords:
    return "darkgreen"
  elif coords in kult_nat_coords:
    return "darkorange"
  elif coords in disharm_harm_coords:
    return "darkred"
  elif coords in krank_gesund_coords:
    return "magenta"
  else:
    return "black" # default: keine codierung zwischen diesen entitäten bzw bei relationen: es handelt sich um eine wertopposition


class Arrow3D(FancyArrowPatch):
    # https://stackoverflow.com/questions/22867620/putting-arrowheads-on-vectors-in-a-3d-plot
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))

        return np.min(zs)

class LayeredNetworkGraph(object):
    # https://stackoverflow.com/questions/60392940/multi-layer-graph-in-networkx
    """
    Plot multi-graphs in 3D.
    """
    def __init__(self, graphs, nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen, alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels=None, layout=nx.spring_layout, ax=None):
        """Given an ordered list of graphs [g1, g2, ..., gn] that represent
        different layers in a multi-layer network, plot the network in
        3D with the different layers separated along the z-axis.

        Within a layer, the corresponding graph defines the connectivity.
        Between layers, nodes in subsequent layers are connected if
        they have the same node ID.

        Arguments:
        ----------
        graphs : list of networkx.Graph objects
            List of graphs, one for each layer.

        node_labels : dict node ID : str label or None (default None)
            Dictionary mapping nodes to labels.
            If None is provided, nodes are not labelled.

        layout_func : function handle (default networkx.spring_layout)
            Function used to compute the layout.

        ax : mpl_toolkits.mplot3d.Axes3d instance or None (default None)
            The axis to plot to. If None is given, a new figure and a new axis are created.

        """
        #New variables
        self.nat_kult_Oppositionen = nat_kult_Oppositionen
        self.alt_neu_Oppositionen = alt_neu_Oppositionen
        self.flach_tief_Oppositionen =flach_tief_Oppositionen
        self.disharm_harm_Oppositionen = disharm_harm_Oppositionen
        self.krank_gesund_Oppositionen = krank_gesund_Oppositionen

        self.alt_neu = alt_neu
        self.flach_tief =flach_tief
        self.kult_nat = kult_nat
        self.disharm_harm = disharm_harm
        self.krank_gesund = krank_gesund

        self.Figuren_pos_bewertet = Figuren_pos_bewertet
        self.Erzähler_pos_bewertet =Erzähler_pos_bewertet
        self.Figuren_neutr_bewertet = Figuren_neutr_bewertet
        self.Erzähler_neutr_bewertet =Erzähler_neutr_bewertet
        self.Figuren_neg_bewertet = Figuren_neg_bewertet
        self.Erzähler_neg_bewertet =Erzähler_neg_bewertet


        # book-keeping
        self.graphs = graphs
        self.total_layers = len(graphs)

        self.node_labels = node_labels
        self.layout = layout

        if ax:
            self.ax = ax
        else:
            fig = plt.figure()
            self.ax = fig.add_subplot(111, projection='3d')

        # create internal representation of nodes and edges
        self.get_nodes()
        self.get_edges_within_layers()
        self.get_edges_between_layers()

        # compute layout and plot
        self.get_node_positions()
        self.draw()


    def get_nodes(self):
        """Construct an internal representation of nodes with the format (node ID, layer)."""
        self.nodes = []
        for z, g in enumerate(self.graphs):
            self.nodes.extend([(node, z) for node in g.nodes()])


    def get_edges_within_layers(self):
        """Remap edges in the individual layers to the internal representations of the node IDs."""
        self.edges_within_layers = []
        for z, g in enumerate(self.graphs):
            self.edges_within_layers.extend([((source, z), (target, z)) for source, target in g.edges()])


    def get_edges_between_layers(self):
        """Determine edges between layers. Nodes in subsequent layers are
        thought to be connected if they have the same ID."""
        self.edges_between_layers = []
        for z1, g in enumerate(self.graphs[:-1]):
            z2 = z1 + 1
            h = self.graphs[z2]
            shared_nodes = set(g.nodes()) & set(h.nodes())
            self.edges_between_layers.extend([((node, z1), (node, z2)) for node in shared_nodes])


    def get_node_positions(self, *args, **kwargs):
        """Get the node positions in the layered layout."""
        # What we would like to do, is apply the layout function to a combined, layered network.
        # However, networkx layout functions are not implemented for the multi-dimensional case.
        # Futhermore, even if there was such a layout function, there probably would be no straightforward way to
        # specify the planarity requirement for nodes within a layer.
        # Therefor, we compute the layout for the full network in 2D, and then apply the
        # positions to the nodes in all planes.
        # For a force-directed layout, this will approximately do the right thing.
        # TODO: implement FR in 3D with layer constraints.

        composition = self.graphs[0]
        for h in self.graphs[1:]:
            composition = nx.compose(composition, h)

        pos = self.layout(composition, k=1.5, *args, **kwargs) # adjust k for distance between nodes

        self.node_positions = dict()
        for z, g in enumerate(self.graphs):
            self.node_positions.update({(node, z) : (*pos[node], z) for node in g.nodes()})



    def draw_nodes(self, nodes, *args, **kwargs):
        """
        Originale Funktion so angepasst, dass auf Figuren- bzw. Erzählerebene pos/neg/beides bzw gar nicht gewertete Entitäten
        mit je unterschiedlichen Knotensymbole angezeigt werden.
        """
        for node in nodes:
          x, y, z = self.node_positions[node] # koordinaten
          ent = self.node_labels[node[0]] # name der entität
          # zuerst Knoten auf Figurenebene eintragen
          if z == 0:
            if ent in self.Figuren_pos_bewertet:
                self.ax.scatter(x, y, z, marker="+", c="#1f77b4", *args, **kwargs)
            elif ent in self.Figuren_neg_bewertet:
                self.ax.scatter(x, y, z, marker="_", c="#1f77b4", *args, **kwargs)
            else:
                self.ax.scatter(x, y, z, marker=".", c="#1f77b4", *args, **kwargs)
          # ... dann auf Erzählerebene
          elif z == 1:
            if ent in self.Erzähler_pos_bewertet:
              self.ax.scatter(x, y, z, marker="+", c="#ff7f0e", *args, **kwargs)
            elif ent in self.Erzähler_neg_bewertet:
              self.ax.scatter(x, y, z, marker="_", c="#ff7f0e", *args, **kwargs)
            else:
              self.ax.scatter(x, y, z, marker=".", c="#ff7f0e", *args, **kwargs)
          # ... schließlich auf Ebene der Relationen
          else:
            """
            # Knoten derjenigen Entitäten, die an Relationen beteiligt sind, die nicht Wertoppositionen sind, anders kennzeichnen
            # aktuell über versch Pfeilfarben gelöst
            if ent in nicht_Wertoppositionen:
              self.ax.scatter(x, y, z, marker="*", c=farbe, *args, **kwargs)
            else:
            """
            self.ax.scatter(x, y, z, marker=".", c="#2ca02c", *args, **kwargs)


    # originale draw function
    def draw_edges(self, edges, *args, **kwargs):
        global global_segments
        segments = [(self.node_positions[source], self.node_positions[target]) for source, target in edges]
        global_segments = segments
        line_collection = Line3DCollection(segments, *args, **kwargs)
        self.ax.add_collection3d(line_collection)


    #created on 16-05-25 for network charts. because previous one was producing error.
    def draw_edges(self, edges, *args, **kwargs):
        global global_segments
        segments = []

        for source, target in edges:
            src = self.node_positions.get(source)
            tgt = self.node_positions.get(target)
            if src is None or tgt is None:
                continue  # Skip if either position is missing

            # Ensure both have 3D coordinates
            if len(src) == 2:
                src = (*src, 0)
            if len(tgt) == 2:
                tgt = (*tgt, 0)

            segments.append((src, tgt))

        global_segments = segments

        if not segments:
            return  # Nothing to draw
        try:
            # Ensure segments are shaped correctly: (N, 2, 3)
            _ = np.array(segments).reshape(-1, 2, 3)
        except Exception as e:
            print("Still bad segment shape:", e)
            return

        line_collection = Line3DCollection(segments, *args, **kwargs)
        self.ax.add_collection3d(line_collection)




    def draw_arrows(self, edges, nodes, *args, **kwargs):
        """
        Adds edges as 3D arrows to graph.

        zeichnet unterschiedliche Relationen (oberste Ebene) und Codierungen
        (unteren zwei Ebenen) in unterschiedlichen Farben

        noch offen/Probleme:
        - nur eine Kante zwischen Knoten in nur einer Farbe, auch wenn es vllt mehrere Wertungen/
        Relationen/Codierungen gibt (z.B. sowohl eine Wert- als auch eine natürlich-kulturell-Opposition). Da aber in diesen
        Fällen mehrere Pfeile übereinander dargestellt werden, sind diese Kanten zumindest dicker.
        -  wenn Entitäten sich selbst werten oder codieren, sind sie aktuell ohne Pfeil im Netzwerk platziert.
        Ggf. den Punkt mit einen Pfeil versehen, der auf sich selbst zeigt (wsl kompliziert)
        - ggf for loops für listengenerierungen noch zsmfassen
        """
        # Listen der an Oppositionen beteiligten Knoten (deren Koordinaten). Exkl Wertoppositionen
        nat_kult_opps_coords = []
        alt_neu_opps_coords = []
        flach_tief_opps_coords = []
        disharm_harm_opps_coords = []
        krank_gesund_opps_coords = []

        # Listen der an Codierungen beteiligten Knoten (jeweils codierende u codierte Entität)
        alt_neu_coords = []
        flach_tief_coords = []
        kult_nat_coords = []
        disharm_harm_coords = []
        krank_gesund_coords = []

        # speichert unter dem Entitätennamen die Koordinaten
        # für jede Ebene eigene Liste, weil ja evtl Entitäten auf allen Ebenen vorkommen
        relationen_nodes = {} # oberste Ebene
        erzähler_nodes = {}
        figuren_nodes = {}
        for node in nodes:
          x, y, z = self.node_positions[node] # koordinaten
          ent = self.node_labels[node[0]] # name der entität
          if z == 2:
            relationen_nodes[ent] = (x, y, z)
          elif z == 1:
            erzähler_nodes[ent] = (x, y, z)
          else:
            figuren_nodes[ent] = (x, y, z)





        # Koordinatenlisten füllen
        for relation in self.nat_kult_Oppositionen:
          nat_kult_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

        for relation in self.alt_neu_Oppositionen:
          alt_neu_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

        for relation in self.flach_tief_Oppositionen:
          flach_tief_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

        for relation in self.disharm_harm_Oppositionen:
          disharm_harm_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

        for relation in self.krank_gesund_Oppositionen:
          krank_gesund_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

        # die Koordinaten der an Codierungen beteiligten Entitäten in unterschiedlichen
        # Listen je nach Codierungsart speichern
        for codierung in self.alt_neu:
          # wenn codierende und codierte entität beide auf einer ebene sind, koordinaten abspeichern, um pfeil dazwischen zu zeichnen
          if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
            alt_neu_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
          if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
            alt_neu_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

        for codierung in self.flach_tief:
          if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
            flach_tief_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
          if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
            flach_tief_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

        for codierung in self.kult_nat:
          if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
            kult_nat_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
          if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
            kult_nat_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

        for codierung in self.disharm_harm:
          if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
            disharm_harm_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
          if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
            disharm_harm_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

        for codierung in self.krank_gesund:
          if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
            krank_gesund_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
          if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
            krank_gesund_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

        for source, target in edges:
          s = self.node_positions[source]
          t = self.node_positions[target]
          #self.ax.plot([s[0], t[0]], [s[1], t[1]], [0,0]) # normale lines
          if s[2] == 2: # oberste Ebene wird separat abgefragt, weil anderer arrowstyle als auf anderen Ebenen
              # Oppositionen mit zwei Pfeilköpfen darstellen
              # Farbe entweder schwarz, wenn Wertopposition, ansonsten je nach Oppositionsart
              farbe = codierung_farbe((s, t), alt_neu_opps_coords, flach_tief_opps_coords, nat_kult_opps_coords, disharm_harm_opps_coords, krank_gesund_opps_coords)
              arrow = Arrow3D([s[0], t[0]], [s[1], t[1]], [s[2],t[2]], arrowstyle="<|-|>", color=farbe, *args, **kwargs)
              self.ax.add_artist(arrow)
          else:
            # Pfeile zwischen Entitäten, zwischen denen Codierung besteht, werden (je nach Codierungsart) in anderen Farben als schwarz dargestellt
            farbe = codierung_farbe((s, t), alt_neu_coords, flach_tief_coords, kult_nat_coords, disharm_harm_coords, krank_gesund_coords)
            arrow = Arrow3D([s[0], t[0]], [s[1], t[1]], [s[2],t[2]], arrowstyle="-|>", color=farbe, *args, **kwargs)
            self.ax.add_artist(arrow)




    def get_extent(self, pad=0.1):
        xyz = np.array(list(self.node_positions.values()))
        xmin, ymin, _ = np.min(xyz, axis=0)
        xmax, ymax, _ = np.max(xyz, axis=0)
        dx = xmax - xmin
        dy = ymax - ymin
        return (xmin - pad * dx, xmax + pad * dx), \
            (ymin - pad * dy, ymax + pad * dy)


    def draw_plane(self, z, *args, **kwargs):
        (xmin, xmax), (ymin, ymax) = self.get_extent(pad=0.1)
        u = np.linspace(xmin, xmax, 10)
        v = np.linspace(ymin, ymax, 10)
        U, V = np.meshgrid(u ,v)
        W = z * np.ones_like(U)
        self.ax.plot_surface(U, V, W, *args, **kwargs)


    def draw_node_labels(self, node_labels, font_size=6, *args, **kwargs):
        for node, z in self.nodes:
            if node in node_labels:
                x = self.node_positions[(node, z)][0]
                y = self.node_positions[(node, z)][1]
                z = self.node_positions[(node, z)][2]

                #ax.text(*self.node_positions[(node, z)], node_labels[node], fontsize=font_size, *args, **kwargs)

                # Text minimal unterm Knoten platzieren
                self.ax.text(x, y, z-0.05, node_labels[node], fontsize=font_size, *args, **kwargs)


    def draw(self):
      self.draw_edges(self.edges_between_layers, color='k', alpha=0.5, linestyle='--', zorder=2) # wenn Knoten auf mehreren Ebenen auftauchen, werden sie mit einer gestrichelten Linie u übereinander dargestellt
      self.draw_arrows(self.edges_within_layers, self.nodes, mutation_scale=15, linestyle='solid', alpha=0.3, zorder=2)
      for z in range(self.total_layers):
        self.draw_plane(z, alpha=0.08, zorder=1)
        self.draw_nodes([node for node in self.nodes if node[1]==z], s=60, zorder=3)

      if self.node_labels:
        self.draw_node_labels(self.node_labels,
                              horizontalalignment='center',
                              verticalalignment='center',
                              zorder=100)






def generate_multilayer_graph(DOCUMENT_TITLE, DISPLAY_CHARTS,G_Fig, G, G2,nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen,alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH):
    # define graphs
    n = 5
    g = nx.erdos_renyi_graph(4*n, p=0.1)
    h = nx.erdos_renyi_graph(3*n, p=0.2)
    i = nx.erdos_renyi_graph(2*n, p=0.4)

    #node_labels = {nn : str(nn) for nn in range(4*n)}

    # Customize the node label size
    label_font_size = 4

    # Customize the edge color and thickness
    edge_color = 'black'
    edge_width = 2.0


    # initialise figure and plot
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(111, projection='3d')

    LayeredNetworkGraph( [G_Fig, G, G2], nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen, alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels=node_labels, ax=ax, layout=nx.spring_layout)
    ax.set_axis_off()



    if (TRANSLATE_GERMAN_GRAPH_TO_ENGLISH):
      # Legende, die die Pfeilfarben erklärt
      alt_patch = mpatches.Patch(color='darkblue', label='traditional-modern encoding')
      flach_patch = mpatches.Patch(color='darkgreen', label='profound-superficial encoding')
      kult_patch = mpatches.Patch(color='darkorange', label='natural-cultural encoding')
      disharm_patch = mpatches.Patch(color="darkred", label='harmonious-disharmonious encoding')
      krank_patch = mpatches.Patch(color="magenta", label='sick-healthy encoding')
      plt.legend(handles=[alt_patch, flach_patch, kult_patch, disharm_patch, krank_patch])
    else:
      # Legende, die die Pfeilfarben erklärt
      alt_patch = mpatches.Patch(color='darkblue', label='alt-neu')
      flach_patch = mpatches.Patch(color='darkgreen', label='flach-tief')
      kult_patch = mpatches.Patch(color='darkorange', label='kulturell-natürlich')
      disharm_patch = mpatches.Patch(color="darkred", label='disharmonisch-harmonisch')
      krank_patch = mpatches.Patch(color="magenta", label='krank-gesund')
      plt.legend(handles=[alt_patch, flach_patch, kult_patch, disharm_patch, krank_patch])




    # Save the figure as an image (png format)
    file_namex = DOCUMENT_TITLE + "_multilayered_graph.png"

    # Define the subfolder path
    #current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # for folder path wrt date
    save_dir = f'multilayered_graphs_{CURRENT_DATE}'  # Create directory name based on time and category
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(save_dir, file_namex)
    plt.savefig(file_path, format='png', dpi=300)

    # nur in colab
    if(DISPLAY_CHARTS):
      plt.show()
    plt.close()

## Network Viz 28-06-2028 (all in one cell to remove errors)

In [258]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from mpl_toolkits.mplot3d.art3d import Patch3DCollection
from matplotlib.patches import Arrow
from matplotlib.collections import PatchCollection
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

import matplotlib.patches as mpatches


In [259]:


def multi_layernetwork_viz_new_version(doc, DOCUMENT_TITLE, DISPLAY_CHARTS, G_Fig, G, G2, nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen, alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels, TRANSLATE_GERMAN_GRAPH_TO_ENGLISH):
  print("multi_layernetwork_viz_new_version")



  def codierung_farbe(coords, alt_neu_coords, flach_tief_coords, kult_nat_coords, disharm_harm_coords, krank_gesund_coords):
    """
    Bestimmt Farbe basierend auf Codierungsart (oder Relationsart).

    coords: Koordinaten von codierender und codierter Entität (oder Koordinaten der zur Relation gehörenden Entitäten)
    alt_neu_coords etc: Listen an Koordinaten von alt/neu codierende u als alt/neu codierte Entität (oder alt_neu_opps_coords)
    Diese Listen werden in der Klasse LayeredNetworkGraph erstellt.

    Achtung: Nach Polarität wird nicht differenziert (würde unübersichtlich).
    D.h. sowohl eine als gesund codierte als auch eine als krank codierte Entität
    hätten einen magenta Pfeil, der auf sie zeigt, weil sie zur selben Codierungsart gehören ("gesund-krank").
    """
    if coords in alt_neu_coords:
      return "darkblue"
    elif coords in flach_tief_coords:
      return "darkolivegreen"
    elif coords in kult_nat_coords:
      return "saddlebrown"
    elif coords in disharm_harm_coords:
      return "yellow"
    elif coords in krank_gesund_coords:
      return "magenta"
    else:
      return "black" # default: keine codierung zwischen diesen entitäten bzw bei relationen: es handelt sich um eine wertopposition


  class Arrow3D(FancyArrowPatch):
      # https://stackoverflow.com/questions/22867620/putting-arrowheads-on-vectors-in-a-3d-plot
      def __init__(self, xs, ys, zs, *args, **kwargs):
          super().__init__((0,0), (0,0), *args, **kwargs)
          self._verts3d = xs, ys, zs

      def do_3d_projection(self, renderer=None):
          xs3d, ys3d, zs3d = self._verts3d
          xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
          self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))

          return np.min(zs)

  class LayeredNetworkGraph(object):
      # https://stackoverflow.com/questions/60392940/multi-layer-graph-in-networkx
      """
      Plot multi-graphs in 3D.
      """
      def __init__(self, graphs, node_labels=None, layout=nx.spring_layout, ax=None):
          """Given an ordered list of graphs [g1, g2, ..., gn] that represent
          different layers in a multi-layer network, plot the network in
          3D with the different layers separated along the z-axis.

          Within a layer, the corresponding graph defines the connectivity.
          Between layers, nodes in subsequent layers are connected if
          they have the same node ID.

          Arguments:
          ----------
          graphs : list of networkx.Graph objects
              List of graphs, one for each layer.

          node_labels : dict node ID : str label or None (default None)
              Dictionary mapping nodes to labels.
              If None is provided, nodes are not labelled.

          layout_func : function handle (default networkx.spring_layout)
              Function used to compute the layout.

          ax : mpl_toolkits.mplot3d.Axes3d instance or None (default None)
              The axis to plot to. If None is given, a new figure and a new axis are created.

          """

          # book-keeping
          self.graphs = graphs
          self.total_layers = len(graphs)

          self.node_labels = node_labels
          self.layout = layout

          if ax:
              self.ax = ax
          else:
              fig = plt.figure()
              self.ax = fig.add_subplot(111, projection='3d')

          # create internal representation of nodes and edges
          self.get_nodes()
          self.get_edges_within_layers()
          self.get_edges_between_layers()

          # compute layout and plot
          self.get_node_positions()
          self.draw()


      def get_nodes(self):
          """Construct an internal representation of nodes with the format (node ID, layer)."""
          self.nodes = []
          for z, g in enumerate(self.graphs):
              self.nodes.extend([(node, z) for node in g.nodes()])


      def get_edges_within_layers(self):
          """Remap edges in the individual layers to the internal representations of the node IDs."""
          self.edges_within_layers = []
          for z, g in enumerate(self.graphs):
              self.edges_within_layers.extend([((source, z), (target, z)) for source, target in g.edges()])


      def get_edges_between_layers(self):
          """Determine edges between layers. Nodes in subsequent layers are
          thought to be connected if they have the same ID."""
          self.edges_between_layers = []
          for z1, g in enumerate(self.graphs[:-1]):
              z2 = z1 + 1
              h = self.graphs[z2]
              shared_nodes = set(g.nodes()) & set(h.nodes())
              self.edges_between_layers.extend([((node, z1), (node, z2)) for node in shared_nodes])


      def get_node_positions(self, *args, **kwargs):
          """Get the node positions in the layered layout."""
          # What we would like to do, is apply the layout function to a combined, layered network.
          # However, networkx layout functions are not implemented for the multi-dimensional case.
          # Futhermore, even if there was such a layout function, there probably would be no straightforward way to
          # specify the planarity requirement for nodes within a layer.
          # Therefor, we compute the layout for the full network in 2D, and then apply the
          # positions to the nodes in all planes.
          # For a force-directed layout, this will approximately do the right thing.
          # TODO: implement FR in 3D with layer constraints.

          composition = self.graphs[0]
          for h in self.graphs[1:]:
              composition = nx.compose(composition, h)

          pos = self.layout(composition, k=1.5, *args, **kwargs) # adjust k for distance between nodes

          self.node_positions = dict()
          for z, g in enumerate(self.graphs):
              self.node_positions.update({(node, z) : (*pos[node], z) for node in g.nodes()})


      def draw_nodes(self, nodes, *args, **kwargs):
          """
          Originale Funktion so angepasst, dass auf Figuren- bzw. Erzählerebene pos/neg/beides bzw gar nicht gewertete Entitäten
          mit je unterschiedlichen Knotensymbole angezeigt werden.
          """
          for node in nodes:
            x, y, z = self.node_positions[node] # koordinaten
            ent = self.node_labels[node[0]] # name der entität
            # zuerst Knoten auf Figurenebene eintragen
            if z == 0:
              if ent in Figuren_pos_bewertet:
                  self.ax.scatter(x, y, z, marker="+", c="#1f77b4", *args, **kwargs)
              elif ent in Figuren_neg_bewertet:
                  self.ax.scatter(x, y, z, marker="_", c="#1f77b4", *args, **kwargs)
              else:
                  self.ax.scatter(x, y, z, marker=".", c="#1f77b4", *args, **kwargs)
            # ... dann auf Erzählerebene
            elif z == 1:
              if ent in Erzähler_pos_bewertet:
                self.ax.scatter(x, y, z, marker="+", c="#ff7f0e", *args, **kwargs)
              elif ent in Erzähler_neg_bewertet:
                self.ax.scatter(x, y, z, marker="_", c="#ff7f0e", *args, **kwargs)
              else:
                self.ax.scatter(x, y, z, marker=".", c="#ff7f0e", *args, **kwargs)
            # ... schließlich auf Ebene der Relationen
            else:
              """
              # Knoten derjenigen Entitäten, die an Relationen beteiligt sind, die nicht Wertoppositionen sind, anders kennzeichnen
              # aktuell über versch Pfeilfarben gelöst
              if ent in nicht_Wertoppositionen:
                self.ax.scatter(x, y, z, marker="*", c=farbe, *args, **kwargs)
              else:
              """
              self.ax.scatter(x, y, z, marker=".", c="#2ca02c", *args, **kwargs)


      # originale draw function
      def draw_edges(self, edges, *args, **kwargs):
          global global_segments
          segments = [(self.node_positions[source], self.node_positions[target]) for source, target in edges]
          global_segments = segments
          print("Draw edges:", len(segments), "segments")
          line_collection = Line3DCollection(segments, *args, **kwargs)
          self.ax.add_collection3d(line_collection)


      def draw_arrows(self, edges, nodes, *args, **kwargs):
          """
          Adds edges as 3D arrows to graph.

          zeichnet unterschiedliche Relationen (oberste Ebene) und Codierungen
          (unteren zwei Ebenen) in unterschiedlichen Farben

          noch offen/Probleme:
          - nur eine Kante zwischen Knoten in nur einer Farbe, auch wenn es vllt mehrere Wertungen/
          Relationen/Codierungen gibt (z.B. sowohl eine Wert- als auch eine natürlich-kulturell-Opposition). Da aber in diesen
          Fällen mehrere Pfeile übereinander dargestellt werden, sind diese Kanten zumindest dicker.
          -  wenn Entitäten sich selbst werten oder codieren, sind sie aktuell ohne Pfeil im Netzwerk platziert.
          Ggf. den Punkt mit einen Pfeil versehen, der auf sich selbst zeigt (wsl kompliziert)
          - ggf for loops für listengenerierungen noch zsmfassen
          """
          # Listen der an Oppositionen beteiligten Knoten (deren Koordinaten). Exkl Wertoppositionen
          nat_kult_opps_coords = []
          alt_neu_opps_coords = []
          flach_tief_opps_coords = []
          disharm_harm_opps_coords = []
          krank_gesund_opps_coords = []

          # Listen der an Codierungen beteiligten Knoten (jeweils codierende u codierte Entität)
          alt_neu_coords = []
          flach_tief_coords = []
          kult_nat_coords = []
          disharm_harm_coords = []
          krank_gesund_coords = []

          # speichert unter dem Entitätennamen die Koordinaten
          # für jede Ebene eigene Liste, weil ja evtl Entitäten auf allen Ebenen vorkommen
          relationen_nodes = {} # oberste Ebene
          erzähler_nodes = {}
          figuren_nodes = {}
          for node in nodes:
            x, y, z = self.node_positions[node] # koordinaten
            ent = self.node_labels[node[0]] # name der entität
            if z == 2:
              relationen_nodes[ent] = (x, y, z)
            elif z == 1:
              erzähler_nodes[ent] = (x, y, z)
            else:
              figuren_nodes[ent] = (x, y, z)

          # Koordinatenlisten füllen
          for relation in nat_kult_Oppositionen:
            nat_kult_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

          for relation in alt_neu_Oppositionen:
            alt_neu_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

          for relation in flach_tief_Oppositionen:
            flach_tief_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

          for relation in disharm_harm_Oppositionen:
            disharm_harm_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

          for relation in krank_gesund_Oppositionen:
            krank_gesund_opps_coords.append((relationen_nodes[relation[0]], relationen_nodes[relation[1]]))

          # die Koordinaten der an Codierungen beteiligten Entitäten in unterschiedlichen
          # Listen je nach Codierungsart speichern
          for codierung in alt_neu:
            # wenn codierende und codierte entität beide auf einer ebene sind, koordinaten abspeichern, um pfeil dazwischen zu zeichnen
            if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
              alt_neu_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
            if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
              alt_neu_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

          for codierung in flach_tief:
            if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
              flach_tief_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
            if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
              flach_tief_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

          for codierung in kult_nat:
            if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
              kult_nat_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
            if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
              kult_nat_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

          for codierung in disharm_harm:
            if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
              disharm_harm_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
            if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
              disharm_harm_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

          for codierung in krank_gesund:
            if codierung[0] in erzähler_nodes.keys() and codierung[1] in erzähler_nodes.keys():
              krank_gesund_coords.append((erzähler_nodes[codierung[0]], erzähler_nodes[codierung[1]]))
            if codierung[0] in figuren_nodes.keys() and codierung[1] in figuren_nodes.keys():
              krank_gesund_coords.append((figuren_nodes[codierung[0]], figuren_nodes[codierung[1]]))

          for source, target in edges:
            s = self.node_positions[source]
            t = self.node_positions[target]
            #self.ax.plot([s[0], t[0]], [s[1], t[1]], [0,0]) # normale lines
            if s[2] == 2: # oberste Ebene wird separat abgefragt, weil anderer arrowstyle als auf anderen Ebenen
                # Oppositionen mit zwei Pfeilköpfen darstellen
                # Farbe entweder schwarz, wenn Wertopposition, ansonsten je nach Oppositionsart
                farbe = codierung_farbe((s, t), alt_neu_opps_coords, flach_tief_opps_coords, nat_kult_opps_coords, disharm_harm_opps_coords, krank_gesund_opps_coords)
                arrow = Arrow3D([s[0], t[0]], [s[1], t[1]], [s[2],t[2]], arrowstyle="<|-|>", color=farbe, *args, **kwargs)
                self.ax.add_artist(arrow)
            else:
              # Pfeile zwischen Entitäten, zwischen denen Codierung besteht, werden (je nach Codierungsart) in anderen Farben als schwarz dargestellt
              farbe = codierung_farbe((s, t), alt_neu_coords, flach_tief_coords, kult_nat_coords, disharm_harm_coords, krank_gesund_coords)
              arrow = Arrow3D([s[0], t[0]], [s[1], t[1]], [s[2],t[2]], arrowstyle="-|>", color=farbe, *args, **kwargs)
              self.ax.add_artist(arrow)




      def get_extent(self, pad=0.1):
          xyz = np.array(list(self.node_positions.values()))
          xmin, ymin, _ = np.min(xyz, axis=0)
          xmax, ymax, _ = np.max(xyz, axis=0)
          dx = xmax - xmin
          dy = ymax - ymin
          return (xmin - pad * dx, xmax + pad * dx), \
              (ymin - pad * dy, ymax + pad * dy)


      def draw_plane(self, z, *args, **kwargs):
          (xmin, xmax), (ymin, ymax) = self.get_extent(pad=0.1)
          u = np.linspace(xmin, xmax, 10)
          v = np.linspace(ymin, ymax, 10)
          U, V = np.meshgrid(u ,v)
          W = z * np.ones_like(U)
          self.ax.plot_surface(U, V, W, *args, **kwargs)


      def draw_node_labels(self, node_labels, font_size=6, *args, **kwargs):
          for node, z in self.nodes:
              if node in node_labels:
                  x = self.node_positions[(node, z)][0]
                  y = self.node_positions[(node, z)][1]
                  z = self.node_positions[(node, z)][2]

                  #ax.text(*self.node_positions[(node, z)], node_labels[node], fontsize=font_size, *args, **kwargs)

                  # Text minimal unterm Knoten platzieren
                  ax.text(x, y, z-0.05, node_labels[node], fontsize=font_size, *args, **kwargs)


      def draw(self):
          self.draw_edges(self.edges_between_layers, color='k', alpha=0.3, linestyle='--', zorder=2) # wenn Knoten auf mehreren Ebenen auftauchen, werden sie mit einer gestrichelten Linie u übereinander dargestellt
          self.draw_arrows(self.edges_within_layers, self.nodes, mutation_scale=15, linestyle='solid', alpha=0.3, zorder=2)

          for z in range(self.total_layers):
              self.draw_plane(z, alpha=0.2, zorder=1)
              self.draw_nodes([node for node in self.nodes if node[1]==z], s=60, zorder=3)

          if self.node_labels:
              self.draw_node_labels(self.node_labels,
                                    horizontalalignment='center',
                                    verticalalignment='center',
                                    zorder=100)




  # define graphs
  n = 5
  g = nx.erdos_renyi_graph(4*n, p=0.1)
  h = nx.erdos_renyi_graph(3*n, p=0.2)
  i = nx.erdos_renyi_graph(2*n, p=0.4)

  #node_labels = {nn : str(nn) for nn in range(4*n)}

  # Customize the node label size
  label_font_size = 4

  # Customize the edge color and thickness
  edge_color = 'black'
  edge_width = 2.0


  # initialise figure and plot
  fig = plt.figure(figsize=(16, 12))
  ax = fig.add_subplot(111, projection='3d')

  LayeredNetworkGraph([G_Fig, G, G2], node_labels=node_labels, ax=ax, layout=nx.spring_layout)
  ax.set_axis_off()

  # Legende, die die Pfeilfarben erklärt
  alt_patch = mpatches.Patch(color='darkblue', label='alt-neu')
  flach_patch = mpatches.Patch(color='darkolivegreen', label='flach-tief')
  kult_patch = mpatches.Patch(color='saddlebrown', label='kulturell-natürlich')
  disharm_patch = mpatches.Patch(color="yellow", label='disharmonisch-harmonisch')
  krank_patch = mpatches.Patch(color="magenta", label='krank-gesund')
  plt.legend(handles=[alt_patch, flach_patch, kult_patch, disharm_patch, krank_patch])

  plt.savefig("temp_network.png", format='png', dpi=300)
  plt.close()





# All functions used are integrated here for convience

In [260]:
def perform_analysis(typesystem_path,cas_file_path,DISPLAY_CHARTS,EXCLUDE_SELF_EVALUATIONS,DEBUG,DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT,SIMPLIFY_GRAPHS,CREATE_SEPARATE_GRAPHS,CREATE_TWO_OPPOSITE_EDGES_OPPOSITION,RECOLOR_TO_GREY_EVALUATIONS,change_some_labels_to_encoding_oppositions,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH,SIMPLIFY_OPPOSITIONS,ANTI_EDGE_OPPOSITION):
  with open(typesystem_path, 'rb') as f:
    typesystem_xml = load_typesystem(f)

  with open(cas_file_path, 'rb') as f:
    doc_CAS_xmi = load_cas_from_xmi(f, typesystem=typesystem_xml) # cassis requires xml 1.0  not 1.1! somehow it seems to work nevertheless?
  DOCUMENT_TITLE = document_metadata_info(typesystem_xml,doc_CAS_xmi,DEBUG)
  print("\nCurrently processing:",DOCUMENT_TITLE)


  RELATIONEN = xRelationen(doc_CAS_xmi, EXCLUDE_SELF_EVALUATIONS,DEBUG)
  #polarität_zu_int(polarität) used within Codierungen
  CODIERUNGEN, j = xCodierungen(doc_CAS_xmi,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH,DEBUG)


  Erzhlerwechsel_dict = xErzhlerwechsel(doc_CAS_xmi,DEBUG)
  WERTUNGEN, i = xWertungen(doc_CAS_xmi, j, Erzhlerwechsel_dict, EXCLUDE_SELF_EVALUATIONS,DEBUG)
  ENT_BEWERT = xPositive_und_negativ_bewertete_Entitäten(WERTUNGEN,DEBUG)
  print("\n")

  #13-12-2024: "nicht spezifiziert" should be treated as "sonstige" (because we dropped this distinction in the annotation)
  WERTUNGEN = nichtspezifiziert_to_sonstige(WERTUNGEN)


  DOCUMENT_TITLE = document_metadata_info(typesystem_xml,doc_CAS_xmi,DEBUG)

  #24-05-2025:MatterMost: Remove
  #if(DOCUMENT_TITLE=="Der neue Advokat"): #18-12-2024:MatterMost
   # DOCUMENT_TITLE= "Kafka_Der_neue_Advokat"
  #if(DOCUMENT_TITLE=="Ein Bericht für eine Akademie"):
   #  DOCUMENT_TITLE="Kafka_Ein_Bericht_für_eine_Akademie"
  #DOCUMENT_TITLE = DOCUMENT_TITLE.replace("_clean.txt", "")
  #DOCUMENT_TITLE = DOCUMENT_TITLE.replace(".txt", "")

  if(change_some_labels_to_encoding_oppositions):
    RELATIONEN = change_opposition_label(RELATIONEN)


  if(TRANSLATE_GERMAN_GRAPH_TO_ENGLISH):
    RELATIONEN, WERTUNGEN = translate_graph_to_english(RELATIONEN,WERTUNGEN) #Added 11-12-2024
  Wertung_und_Oppositionen_als_Kanten(WERTUNGEN, RELATIONEN, ENT_BEWERT, DOCUMENT_TITLE, DISPLAY_CHARTS,DEBUG,DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT,SIMPLIFY_GRAPHS,CREATE_SEPARATE_GRAPHS,CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION,RECOLOR_TO_GREY_EVALUATIONS,SIMPLIFY_OPPOSITIONS,ANTI_EDGE_OPPOSITION)

  DF = xCodierungen_und_oder_Wertungen_anzeigen(CODIERUNGEN, WERTUNGEN,  DEBUG, anzeige_option = "y")
  G, G_FIG = xCodierungen_und_Wertungen_als_Kanten(DF,DOCUMENT_TITLE,DISPLAY_CHARTS)



  #G_FIG =
  xFigurenwertungen(G_FIG, DOCUMENT_TITLE,DISPLAY_CHARTS)
  G2 = xNetzwerk_mit_annotierten_oppositionen_als_kanten(RELATIONEN, i, DOCUMENT_TITLE,DISPLAY_CHARTS)




  #16-05-2025 removed condition since now it is working with multi layer network. but have to check again.
  #if TRANSLATE_GERMAN_GRAPH_TO_ENGLISH: #multi layer does not work on translation so not added in above condition.
  #Following is only for multilayer_graph
  nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen = update_Relationen_oppositions(RELATIONEN)
  alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund = update_Codierungen_categories(CODIERUNGEN)
  Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet = update_bewertet_categories(ENT_BEWERT)

  node_labels = update_nodelabels(G, G_FIG, G2,DEBUG)


  # This is new version to remove errors. , below code could be removed after confirmation
  #  multi_layernetwork_viz_new_version(doc_CAS_xmi,DOCUMENT_TITLE, DISPLAY_CHARTS, G_FIG, G, G2, nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen, alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels, TRANSLATE_GERMAN_GRAPH_TO_ENGLISH)



  generate_multilayer_graph(DOCUMENT_TITLE, DISPLAY_CHARTS, G_FIG, G, G2, nat_kult_Oppositionen ,  alt_neu_Oppositionen,  flach_tief_Oppositionen,  disharm_harm_Oppositionen,  krank_gesund_Oppositionen, alt_neu, flach_tief ,kult_nat,disharm_harm,krank_gesund,Figuren_pos_bewertet, Erzähler_pos_bewertet, Figuren_neutr_bewertet,Erzähler_neutr_bewertet, Figuren_neg_bewertet,Erzähler_neg_bewertet,node_labels, TRANSLATE_GERMAN_GRAPH_TO_ENGLISH)



# Main cell to call the funtions

In [261]:
'''
For optimal visualization. Last tested on 20-05-2025

EXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.
DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen
CREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.

#Merging for the evaluations and oppositions
SIMPLIFY_EVALUATIONS = False #24-11-2024: one edge for each source target for each category for Evaluations.-
SIMPLIFY_OPPOSITIONS = False #18-12-2024: one edge for each source target for each category for Oppositions.
CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).

if(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION): #when Two edge for oppositions is required then SIMPLIFY_GRAPHS is to be set true for evaluations.
  SIMPLIFY_OPPOSITIONS=True


ANTI_EDGE_OPPOSITION = True #06-01-2025 this creates reverse opposition edge for all oppositions.


RECOLOR_TO_GREY_EVALUATIONS = False #26-11-2024: Recolor evaluative_network_with_oppositions to grey and orange only instead of red,green and orange.
TRANSLATE_GERMAN_GRAPH_TO_ENGLISH = True #12-12-2024
change_some_labels_to_encoding_oppositions=True #26-01-2025 changes some relation labels to encoding labels.
'''

'\nFor optimal visualization. Last tested on 20-05-2025\n\nEXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.\nDROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen\nCREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.\n\n#Merging for the evaluations and oppositions\nSIMPLIFY_EVALUATIONS = False #24-11-2024: one edge for each source target for each category for Evaluations.-\nSIMPLIFY_OPPOSITIONS = False #18-12-2024: one edge for each source target for each category for Oppositions.\nCREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).\n\nif(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION): #when Two edge for oppositions is required then SIMPLIFY_GRAPHS is to be set true for e

In [262]:
'''
#Graphlet settings
set_opposition_weights = True #17-06-2025 Must be set true now. if not then other parts need modifiation to set default weight 0

EXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.
DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen
CREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.

#Merging for the evaluations and oppositions
SIMPLIFY_EVALUATIONS = True #24-11-2024: one edge for each source target for each category for Evaluations.-
SIMPLIFY_OPPOSITIONS = True #18-12-2024: one edge for each source target for each category for Oppositions.
CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).

if(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION): #when Two edge for oppositions is required then SIMPLIFY_GRAPHS is to be set true for evaluations.
  SIMPLIFY_OPPOSITIONS=True


ANTI_EDGE_OPPOSITION = True #06-01-2025 this creates reverse opposition edge for all oppositions.


RECOLOR_TO_GREY_EVALUATIONS = False #26-11-2024: Recolor evaluative_network_with_oppositions to grey and orange only instead of red,green and orange.
TRANSLATE_GERMAN_GRAPH_TO_ENGLISH = True #12-12-2024
change_some_labels_to_encoding_oppositions=True #26-01-2025 changes some relation labels to encoding labels.
'''


'\n#Graphlet settings\nset_opposition_weights = True #17-06-2025 Must be set true now. if not then other parts need modifiation to set default weight 0\n\nEXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.\nDROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen\nCREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.\n\n#Merging for the evaluations and oppositions\nSIMPLIFY_EVALUATIONS = True #24-11-2024: one edge for each source target for each category for Evaluations.-\nSIMPLIFY_OPPOSITIONS = True #18-12-2024: one edge for each source target for each category for Oppositions.\nCREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).\n\nif(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSI

In [ ]:
'''
#For multilayered. Last tested on 29-05-2025
set_opposition_weights = True #17-06-2025 Must be set true now. if not then other parts need modifiation to set default weight 0
abs_polaritPolarität = False #17-06-2025 positive and negative as +1 for evaluiations
abs_polaritPolarität = False #17-06-2025 positive and negative as +1 for evaluiations
split_pos_neg_evalustion_graphs = False #17-06-2025 positive and negative as seperate graphs for evaluiations
if(split_pos_neg_evalustion_graphs):
  abs_polaritPolarität=False

EXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.
DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen
CREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.

#Merging for the evaluations and oppositions
SIMPLIFY_EVALUATIONS = False #24-11-2024: one edge for each source target for each category for Evaluations.-
SIMPLIFY_OPPOSITIONS = False #18-12-2024: one edge for each source target for each category for Oppositions.
CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).

if(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION): #when Two edge for oppositions is required then SIMPLIFY_GRAPHS is to be set true for evaluations.
  SIMPLIFY_OPPOSITIONS=True


ANTI_EDGE_OPPOSITION = False #06-01-2025 this creates reverse opposition edge for all oppositions.


RECOLOR_TO_GREY_EVALUATIONS = False #26-11-2024: Recolor evaluative_network_with_oppositions to grey and orange only instead of red,green and orange.
TRANSLATE_GERMAN_GRAPH_TO_ENGLISH = True #12-12-2024
change_some_labels_to_encoding_oppositions= True #26-01-2025 changes some relation labels to encoding labels.
'''


In [264]:
from google.colab import files #27-11-2024: After using zip function below it needs to be imported again for unknown reasons.

DOCUMENT_LIST_RELATIONEN = []
DOCUMENT_LIST_WERTUNGEN = []
DOCUMENT_LIST_WERTUNGEN_UND_RELATIONEN = []



network_list = []

network_list_Wertungen = []
network_list_Relationen = []
network_list_Wertungen_und_Relationen = []

DOCUMENT_TITLES = []

#Main cell for getting and analyzing data from annotated files

# Path to the folder in Google Drive where the files are stored
drive_folder_path = '/content/'

#Rilke_Die_Weise_von_Liebe_und_Tod_des_Cornets_Christoph_Rilke_clean.txt
#Hollenstein_Gelb_wie_eine_Zitrone.txt



#For multilayered. Last tested on 29-05-2025
set_opposition_weights = True #17-06-2025 Must be set true now. if not then other parts need modifiation to set default weight 0
abs_polaritPolarität = False #17-06-2025 positive and negative as +1 for evaluiations
abs_polaritPolarität = False #17-06-2025 positive and negative as +1 for evaluiations
split_pos_neg_evalustion_graphs = False #17-06-2025 positive and negative as seperate graphs for evaluiations
if(split_pos_neg_evalustion_graphs):
  abs_polaritPolarität=False

EXCLUDE_SELF_EVALUATIONS = False #17-11-2024 Exlcudes self evaluations for the network graph.
DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT = True #24-11-2024 remove element von and Zugehörigkeit from Relationen
CREATE_SEPARATE_GRAPHS = False #24-11-2024: create seperate graphs for Evaluation and Oppositions.

#Merging for the evaluations and oppositions
SIMPLIFY_EVALUATIONS = False #24-11-2024: one edge for each source target for each category for Evaluations.-
SIMPLIFY_OPPOSITIONS = False #18-12-2024: one edge for each source target for each category for Oppositions.
CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION = False #(NOT NEEDED ANY MORE) 25-11-2024an option that creates for all oppositions two edges instead of one, one in one direction and another in the opposite direction (with the same label).

if(CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION): #when Two edge for oppositions is required then SIMPLIFY_GRAPHS is to be set true for evaluations.
  SIMPLIFY_OPPOSITIONS=True


ANTI_EDGE_OPPOSITION = False #06-01-2025 this creates reverse opposition edge for all oppositions.


RECOLOR_TO_GREY_EVALUATIONS = False #26-11-2024: Recolor evaluative_network_with_oppositions to grey and orange only instead of red,green and orange.
TRANSLATE_GERMAN_GRAPH_TO_ENGLISH = True #12-12-2024
change_some_labels_to_encoding_oppositions= True #26-01-2025 changes some relation labels to encoding labels.






#24-11-2024: one edge for each source target for each category for Evaluations.
SIMPLIFY_GRAPHS = SIMPLIFY_EVALUATIONS #18-12-2024: one edge for each source target for each category for Evaluations.

#PROCESS_SINGLE_AUTHOR = True

# delete all files
action_delete_all_files(drive_folder_path)



if PROCESS_SINGLE_AUTHOR:
  print('upload inception annotated zip of SINGLE author')
  upload_next_author_zip()

  print('\n')

  # Reading a Common Analysis System (CAS) file
  typesystem_path = drive_folder_path + "TypeSystem.xml"
  cas_file_path = drive_folder_path + "CURATION_USER.xmi"

  perform_analysis(typesystem_path,cas_file_path,DISPLAY_CHARTS,EXCLUDE_SELF_EVALUATIONS,DEBUG,DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT,SIMPLIFY_GRAPHS,CREATE_SEPARATE_GRAPHS,CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION,RECOLOR_TO_GREY_EVALUATIONS,change_some_labels_to_encoding_oppositions,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH,SIMPLIFY_OPPOSITIONS,ANTI_EDGE_OPPOSITION)

else:
  print('upload inception annotated zip containing ALL authors')
  #Upload the ZIP file
  uploaded_zip = files.upload()

  # Extract the filename from the uploaded files
  zip_filename = list(uploaded_zip.keys())[0]

  # Extraction path
  extract_path = './'

  extract_nested_zip(zip_filename,'./')

  # Extract files
  extract_nested_zip(zip_filename, extract_path)

  # Get folder names
  folders = get_folder_names()

  for folder in folders:
    # Reading a Common Analysis System (CAS) file
    typesystem_path = drive_folder_path + 'curation/' + folder + "/TypeSystem.xml"
    cas_file_path = drive_folder_path + 'curation/' + folder + "/CURATION_USER.xmi"
    perform_analysis(typesystem_path,cas_file_path,DISPLAY_CHARTS,EXCLUDE_SELF_EVALUATIONS,DEBUG,DROP_ELEMENT_VON_AND_ZÜGEHORIGKEIT,SIMPLIFY_GRAPHS,CREATE_SEPARATE_GRAPHS,CREATE_TWO_OPPOSITE_EDGES_FOR_OPPOSITION,RECOLOR_TO_GREY_EVALUATIONS,change_some_labels_to_encoding_oppositions,TRANSLATE_GERMAN_GRAPH_TO_ENGLISH,SIMPLIFY_OPPOSITIONS,ANTI_EDGE_OPPOSITION)


☠️ Deleted directory: /content/complete_network_28Jun2025
☠️ Deleted directory: /content/curation
☠️ Deleted directory: /content/complete_network_graphml_28Jun2025
☠️ Deleted directory: /content/multilayered_graphs_28Jun2025
☠️ Deleted file: /content/vis_test - 1.zip
☠️ Deleted directory: /content/Basic_narrator_evaluations_G_28Jun2025
☠️ Deleted directory: /content/Basic_character_evaluations_G_Fig_nx_28Jun2025
☠️ Deleted directory: /content/Basic_opposition_networks_G2_nx_28Jun2025
upload inception annotated zip containing ALL authors


Saving curated-docs-evaluative-structures-and-cultural-cri-1-2025-05-26-132454.zip to curated-docs-evaluative-structures-and-cultural-cri-1-2025-05-26-132454.zip

Currently processing: Grimm_Das_tapfere_Schneiderlein.txt


============Relationen============================ {1: ('unsere Höhle', 'die Werkstätte', 'Feature Opposition', 1), 2: ('eine besondere Wohnung', 'die Werkstätte', 'Feature Opposition', 1), 3: ('ein Schneiderlein', 'ein gewaltiger Riese', 'Value Opposition', 1), 4: ('ein Schneiderlein', 'die Kriegsleute', 'Feature Opposition', 1)}
complete_network_29Jun2025/Grimm_Das_tapfere_Schneiderlein.txt_complete_network.html
Basic_narrator_evaluations_G_29Jun2025/Grimm_Das_tapfere_Schneiderlein.txt_G_nx.html
Basic_character_evaluations_G_Fig_nx_29Jun2025/Grimm_Das_tapfere_Schneiderlein.txt_G_Fig_nx.html
Basic_opposition_networks_G2_nx_29Jun2025/Grimm_Das_tapfere_Schneiderlein.txt_G2_nx.html
def update_Relationen_oppositions(Relationen): Keine existierende Opposition: %s Feature

# To be used in the Directed_Graphlet_Counter_v3 cpp file.

In [ ]:
import os
import networkx as nx

_current_date = CURRENT_DATE  # For folder path wrt date

# Specify the folder containing GraphML files
input_folder = "complete_network_graphml_"+_current_date  # Replace with the path to your folder
output_folder = "complete_network_nodeNum_targetNum_"+_current_date  # Replace with the desired output folder

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Process all .graphml files in the input folder
for filename in os.listdir(input_folder):
    if filename.endswith(".graphml"):
        input_file = os.path.join(input_folder, filename)
        output_file = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.txt")

        # Read the GraphML file
        graph = nx.read_graphml(input_file)

        # Map nodes to integer IDs
        node_mapping = {node: idx for idx, node in enumerate(graph.nodes)}

        # Write edges to the output file
        with open(output_file, "w") as f:
            for edge in graph.edges(data=False):  # Extract edges without attributes
                source = edge[0]
                target = edge[1]
                source_id = node_mapping[source]
                target_id = node_mapping[target]
                f.write(f"{source_id} {target_id}\n")

        print(f"Edges with numeric nodes saved to {output_file}")

# some useful code snippets

In [265]:
#Delete some Extra folders and files within, just leaving "evaluative_network_with_oppositions" folder.
import shutil

unwanted_folders=["curation",
                  #"Basic_character_evaluations_G_Fig_nx_",
                  #"Basic_narrator_evaluations_G_",
                  #"Basic_opposition_networks_G2_nx_",
                  #"split_network_graphml_Relationen_",
                  #"split_network_graphml_Wertungen_",
                  #"multilayered_graphs_",
                  "complete_network_graphml_",
                  #"complete_network_"
                  ]

for folder in unwanted_folders:
  if(folder == "curation"):
    _current_date = ""
  else:
    _current_date = CURRENT_DATE  # for folder path wrt date

  folder_path = '/content/'+folder+_current_date  # Replace with the path to your folder

  # Delete the folder and all its contents
  try:
    shutil.rmtree(folder_path)
  except:
    pass
  print(f"Folder '{folder_path}' and all its contents have been deleted.")

Folder '/content/curation' and all its contents have been deleted.
Folder '/content/complete_network_graphml_29Jun2025' and all its contents have been deleted.


In [267]:
#For Making zip files for all folders except curation.

# Define the path to the parent directory containing the folders
parent_dir = '/content/'  # Target parent directory
tz_Berlin = timezone('Europe/Berlin')
current_date = datetime.now(tz_Berlin).strftime('%Y%m%d')  # For folder path wrt date

# Define the output zip file path (with .zip extension)
output_filename = f'/content/MoL_network_visualization_output_{current_date}.zip'



# list of possible folders
folders_to_zip = [
    "complete_network_",
    "Basic_character_evaluations_G_Fig_nx_",
    "Basic_narrator_evaluations_G_",
    "Basic_opposition_networks_G2_nx_",
    "multilayered_graphs_",
    "split_network_Relationen_",
    "split_network_Wertungen_",
    "split_network_graphml_Relationen_",
    "split_network_graphml_Wertungen_",
    "complete_network_graphml_",
    "complete_network_nodeNum_targetNum_",
    "split_network_Wertungen_abs_negative_",
    "split_network_Wertungen_positive_",
    "split_network_graphml_Wertungen_abs_negative_",
    "split_network_graphml_Wertungen_positive_"
    ]

# Get the current date
_current_date = CURRENT_DATE

# Append the current date to each element in the list
folders_to_zip_with_date = [folder + _current_date for folder in folders_to_zip]

# Create a list of folders that exist
folders_existing = [folder for folder in folders_to_zip_with_date if os.path.exists(os.path.join(parent_dir, folder))]

if folders_existing:
    print(f"Existing folders to zip: {folders_existing}")

    # Create the zip file
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in folders_existing:
            folder_path = os.path.join(parent_dir, folder)
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Add files to the zip, preserving their relative paths
                    arcname = os.path.relpath(file_path, parent_dir)
                    zipf.write(file_path, arcname)

    print(f"Zipped folders saved as {output_filename}")
else:
    print("No folders to zip.")

Existing folders to zip: ['complete_network_29Jun2025', 'Basic_character_evaluations_G_Fig_nx_29Jun2025', 'Basic_narrator_evaluations_G_29Jun2025', 'Basic_opposition_networks_G2_nx_29Jun2025', 'multilayered_graphs_29Jun2025']
Zipped folders saved as /content/MoL_network_visualization_output_20250629.zip
